In [1]:
#%%time
import os
import re
import gzip
import math
import random
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad

from ast import literal_eval
import warnings
# 忽略 FutureWarning 类型警告
warnings.simplefilter(action='ignore', category=FutureWarning)
# 忽略特定类型的警告：忽略 scanpy包中含有 ignore 的 UserWarning 类型警告
warnings.filterwarnings("ignore", category=UserWarning, module="scanpy")
# 禁用 pandas 包中的 SettingWithCopyWarning 类型警告  
pd.options.mode.chained_assignment = None  # 或 'raise' 表示引发异常

import inferECC
from inferECC import *

species value: hg38
species == 'mm10':  False


In [143]:
### anno ecdna2gene oncogene pathway
### import ecdna2gene2pathway function

import warnings
import scanpy as sc
from ast import literal_eval
from collections import Counter

cancer_gene_tb = pd.read_csv("D:/02.project/18.ecDNA/02.method/ref/hg19/cancer_gene_census_20240307.txt",sep="\t")
cancer_gene_list = list(cancer_gene_tb["Gene Symbol"].unique())

# DataFrame :: gene_unique_list is the column list of elements that match oncogenes
def custom_transform(element):
    if isinstance(element, list):
        # Extract the intersection of the element list with oncogene_list
        return list(set(element) & set(cancer_gene_list))
    else:
        # If the element is NaN, then return NaN
        return element
### step 1
# Apply the custom function custom_transform to the gene_unique_list column
#df['oncogene'] = df['gene_unique_list'].apply(custom_transform)

# Define a function to process elements in the oncogene column:: convert oncogene_nor element value 0 to []
def process_gene_list(item):
    # If the element is the integer 0, return an empty list
    if item == 0:
        return []
    # Otherwise, return the original element
    else:
        return item
# Define a function to process elements in the oncogene_nor column:: merge oncogene_nor list
def merge_gene_list(gene_list):
    # Create an empty list to store the results
    result = []
    for item in gene_list:
        # If the element is a list, add its elements to the result
        if isinstance(item, list):
            result.extend(item)
    return result
# Define a function to process elements in the oncogene_nor column
def process_oncogene_nor(oncogene_nor, top_n=15):
    # Use Counter to count the frequency of elements
    counter = Counter(oncogene_nor)
    # Extract the top_n elements with the highest frequency
    most_common = counter.most_common(top_n) # 6/15
    # Form a list with these top_n elements
    result = [item[0] for item in most_common]
    return result
### Main function merge
def group_merge(df=pd.DataFrame(), group_columns_name="group", raw_columns_name="oncogene"):
    df_ecdna_gene = df.copy()
    # Use the apply function
    df_ecdna_gene[raw_columns_name+"_nor"] = df_ecdna_gene[raw_columns_name].apply(process_gene_list)
    # Use groupby and apply functions
    df_ecdna_gene_grouped = df_ecdna_gene.groupby(group_columns_name)[raw_columns_name+"_nor"].apply(merge_gene_list).reset_index()
    # Convert the label column to Categorical type and specify the order of categories
    df_ecdna_gene_grouped[group_columns_name] = pd.Categorical(df_ecdna_gene_grouped[group_columns_name],
                                                    categories=list(df_ecdna_gene[group_columns_name].unique()),
                                                    ordered=True)
    # Use the sort_values function to sort
    df_ecdna_gene_grouped = df_ecdna_gene_grouped.sort_values(group_columns_name)
    # Use the apply function
    df_ecdna_gene_grouped['most_'+raw_columns_name+"_nor"] = df_ecdna_gene_grouped[raw_columns_name+"_nor"].apply(process_oncogene_nor)
    return df_ecdna_gene_grouped

### step 2
# Apply the custom function group_merge to the gene oncogene column
#df_merge = group_merge(df_ecdna_gene_oncogene, "group")



###共性基因功能注释：：
# highest_expr_top_n
import gseapy as gp
from gseapy.plot import barplot, dotplot
"""
names = gp.get_library_name()
# Create a DataFrame from the list
df = pd.DataFrame(names, columns=['Name'])
# Save the DataFrame to a CSV file
df.to_csv("./fig/gseapy_gene-sets.txt", index=False)
"""

import pandas as pd
# 定义一个函数来处理字符串
def process_term(term):
    # 检查倒数第12个到倒数第9个字符是否为'(GO:'
    if term[-12:-8] == '(GO:':
        # 如果是，则删除倒数第11个字符之后的所有字符
        return term[:-12]
    else:
        # 否则，返回原字符串
        return term

def gp_enrichr(gene_list,sample,path,gene_sets):
    # 假设你已经有了差异基因列表
    diff_genes = gene_list
    #diff_genes = new_df[new_df["cluster"]=="C3"]["gene"].to_list()
    #diff_genes = new_df["gene"].to_list()
    # 运行GSEA
    # enrichr库包含了大量的基因集，选择其中的一个，例如'KEGG_2019_Human'\'MSigDB_Hallmark_2020'
    enr = gp.enrichr(gene_list=diff_genes,
                     gene_sets=['MSigDB_Hallmark_2020',
                                #'KEGG_2021_Human',
                                #'GO_Biological_Process_2023',
                                #'GO_Molecular_Function_2023',
                                #'GO_Cellular_Component_2023'
                               ],
                     #outdir='./fig/pdac/gseapy',
                     outdir=None,
                     cutoff=0.05)
    
    enr.results["Term_raw"]=enr.results["Term"]
    # 使用apply函数应用到'Term'列
    enr.results["Term"] = enr.results["Term"].apply(process_term)
    # trim (go:...)
    #enr.results["Term"] = enr.results["Term"].str.split(" \(GO").str[0]
    # 根据'A'列的值对df进行排序
    enr_results_sorted = enr.results.sort_values(by='P-value')
    enr_results_sorted['cluster'] = sample
    enr_results_sorted.to_csv(path+"/"+sample+"_enrichr_result.tsv",sep="\t",index=True)
    
    # categorical scatterplot
    #ax = barplot(enr_results_sorted,
    #             #column="Adjusted P-value",
    #             column="P-value",
    #             #group='Gene_set', # set group, so you could do a multi-sample/library comparsion
    #             #size=10,
    #             top_term=30,
    #             #figsize=(10,6*len(enr.results[enr.results["P-value"] <= 0.05])/50),
    #             figsize=(35,12),
    #             color=['blue'] # set colors for group
    #             #color = {'KEGG_2021_Human': 'salmon', 'MSigDB_Hallmark_2020':'darkblue'}
    #             )
    #plt.grid(False)
    #plt.savefig(path+"/"+sample+"_enrichr_barplot.pdf")
    return enr_results_sorted

In [62]:
import pandas as pd
from pyliftover import LiftOver

# 初始化 liftover 对象
lo = LiftOver('hg19', 'hg38')  # 会自动下载 chain 文件

# 示例 DataFrame（请替换为实际数据）
df = pd.DataFrame({
    'chr_100k': ['chr1:0-100000', 'chr2:100000-200000', 'chrY:59000000-59100000', 'chrY:59000000-159100000']
})

# 定义转换函数
def convert_interval(interval):
    try:
        chrom, pos_range = interval.split(':')
        start_str, end_str = pos_range.replace('_', '-').split('-')
        start, end = int(start_str), int(end_str)
        length = end - start

        # 起点不能为0，改为1尝试
        start_to_convert = 1 if start == 0 else start

        # 尝试 liftover 转换
        res_start = lo.convert_coordinate(chrom, start_to_convert)
        res_end = lo.convert_coordinate(chrom, end - 1)

        # 情况1：两者都失败
        if not res_start and not res_end:
            return None

        # 情况2：start 失败，end 成功
        elif not res_start and res_end:
            new_chrom = res_end[0][0]
            new_end = res_end[0][1] + 1
            new_start = max(new_end - length, 0)
            return f"{new_chrom}:{int(new_start)}_{int(new_end)}"

        # 情况3：end 失败，start 成功
        elif res_start and not res_end:
            new_chrom = res_start[0][0]
            new_start = res_start[0][1]
            new_end = new_start + length
            return f"{new_chrom}:{int(new_start)}_{int(new_end)}"

        # 情况4：两者都成功
        else:
            new_chrom = res_start[0][0]
            new_start = res_start[0][1]
            new_end = res_end[0][1] + 1
            return f"{new_chrom}:{int(new_start)}_{int(new_end)}"

    except Exception as e:
        print(f"Error processing interval {interval}: {e}")
        return None

# 应用转换函数
df['chr_100k_hg38'] = df['chr_100k'].apply(convert_interval)

# 输出结果
print(df)


                  chr_100k            chr_100k_hg38
0            chr1:0-100000            chr1:0_100000
1       chr2:100000-200000       chr2:100000_200000
2   chrY:59000000-59100000   chrY:56853853_56953852
3  chrY:59000000-159100000  chrY:56853853_156953853


In [3]:
####### 
# 输入数据库 ECDNA 做 OVERLAP
# 数据清洗 分类标签一致

In [ ]:
####### database 1：：circlebase

In [192]:
from pathlib import Path
import pandas as pd
import numpy as np

# 文件路径
file_path = Path(r"E:\05.project\04.ecDNA\01.data\ECDNA-DB\归档\circlebase.Cancer_tissue.tsv")

# 读取 TSV 文件
circlebase = pd.read_csv(file_path, sep='\t')

# 显示原始行数
print("原始数据行数：", len(circlebase))

# 统计 NaN 和 Inf 的数量
nan_counts = circlebase[["start_hg38", "end_hg38"]].isna().sum()
inf_counts = ((circlebase[["start_hg38", "end_hg38"]] == np.inf) | (circlebase[["start_hg38", "end_hg38"]] == -np.inf)).sum()

print("NaN counts:\n", nan_counts)
print("Inf counts:\n", inf_counts)

# 清理数据：将 inf 替换为 NaN，然后丢弃包含 NaN 的行
circlebase_clean = circlebase.copy()
circlebase_clean[["start_hg38", "end_hg38"]] = circlebase_clean[["start_hg38", "end_hg38"]].replace([np.inf, -np.inf], np.nan)
circlebase_clean = circlebase_clean.dropna(subset=["start_hg38", "end_hg38"])

# 显示清理后的行数
print("清理后数据行数：", len(circlebase_clean))

# 转换为整数类型
circlebase_clean["start_hg38"] = circlebase_clean["start_hg38"].astype(int)
circlebase_clean["end_hg38"] = circlebase_clean["end_hg38"].astype(int)

# 拼接为 ecdna_region 列
circlebase_clean["ecdna_region_hg38"] = (
    circlebase_clean["chr_hg38"] + ":" +
    circlebase_clean["start_hg38"].astype(str) + "_" +
    circlebase_clean["end_hg38"].astype(str)
)

# 预览前几行结果
print(circlebase_clean[["chr_hg38", "start_hg38", "end_hg38", "ecdna_region_hg38"]].head())

原始数据行数： 6498
NaN counts:
 start_hg38    198
end_hg38      198
dtype: int64
Inf counts:
 start_hg38    0
end_hg38      0
dtype: int64
清理后数据行数： 6300
  chr_hg38  start_hg38   end_hg38         ecdna_region_hg38
0     chr1   203704017  204689872  chr1:203704017_204689872
1     chr7    54836197   55242033    chr7:54836197_55242033
2    chr20    25252349   25252610   chr20:25252349_25252610
3     chr1    40138329   41623929    chr1:40138329_41623929
4     chr1    45419329   46091328    chr1:45419329_46091328


In [193]:
circlebase_clean["sample_type"].value_counts()

Sarcoma                              931
Breast cancer                        695
Esophageal cancer                    622
Glioblastoma                         619
Gastric cancer                       418
Skin cancer                          418
Lung adenocarcinoma                  370
Bladder cancer                       353
Ovarian cancer                       316
Head and neck cancer                 294
Uterine corpus endometrial cancer    242
Lung Squamous cell cancer            217
Pancreatic cancer                    144
Liver cancer                         138
Lower grade glioma cancer             99
Prostate cancer                       79
Colorectal cancer                     76
Ewing Sarcoma                         60
Renal cancer                          60
Cervical cancer                       49
Breast cancer/Brain Met               28
Pediatric Brain cancer                17
B-cell lymphoma                       10
Medulloblastoma                        8
Uveal melanoma  

In [194]:
# 原始自定义映射（用户已确认）
manual_map = {
    "Breast cancer": "BRCA",
    "Cervical cancer": "CESC",
    "Colorectal cancer": "CRC",
    "Colon cancer": "CRC",
    "Head and neck cancer": "HNSCC",
    "Pancreatic cancer": "PDAC",
    "Ovarian cancer": "OV",
    "Skin cancer": "SKCM",
    "Uterine corpus endometrial cancer": "UCEC",
    "Breast cancer/Brain Met": "BRCA",
    "Glioblastoma\xa0":"GBM",
    "Glioblastoma?": "GBM",
    "Glioblastoma": "GBM",
    "Lower grade glioma cancer": "LGG",
    "Lung adenocarcinoma": "LUAD",
    "Lung Squamous cell cancer": "LUSC",
    "Lung cancer/Brain Met": "LUAD",
    "Liver cancer": "LIHC",
    "Esophageal cancer": "ESCA",
    "Gastric cancer": "STAD",
    "Sarcoma": "SARC",
    "Bladder cancer": "BLCA",
    "Prostate cancer": "PRAD",
    "Renal cancer": "KIRC",
    "Ewing Sarcoma": "EWS",
    "Pediatric Brain cancer": "PBCA",
    "B-cell lymphoma": "BCL",
    "Oral cancer": "ORAL",
    "Medulloblastoma": "MB",
    "Uveal melanoma": "UVM",
    "Biliary tract cancer": "CHOL",
    "Brain": "BRAIN"
}

# 应用映射替换
circlebase_clean["sample_type_abbr"] = circlebase_clean["sample_type"].map(manual_map)

# 若有未覆盖项，保留原始名称以便后续人工审查
circlebase_clean["sample_type_abbr"] = circlebase_clean["sample_type_abbr"].fillna(circlebase["sample_type"])

# 查看替换结果的唯一值列表（可用于检查是否仍有未缩写的项目）
print(circlebase_clean["sample_type_abbr"].value_counts())

SARC     931
BRCA     723
GBM      625
ESCA     622
STAD     418
SKCM     418
LUAD     376
BLCA     353
OV       316
HNSCC    294
UCEC     242
LUSC     217
PDAC     144
LIHC     138
LGG       99
PRAD      79
CRC       77
EWS       60
KIRC      60
CESC      49
PBCA      17
BCL       10
UVM        8
MB         8
CHOL       7
ORAL       7
BRAIN      2
Name: sample_type_abbr, dtype: int64


In [ ]:
####### database 2：：eccDNA_Atlas

In [4]:
eccDNA_Atlas = pd.read_csv("E:/05.project/04.ecDNA/01.data/ECDNA-DB/eccDNA Atlas.txt/eccDNA Atlas.txt", sep="\t",low_memory=False)
eccDNA_Atlas

,Species,eccDNA type,eccDNA ID,Genome version,Location,Chr,Start,End,Health/Disease,Tissue/Cell line,Disease Name,Experiment/Prediction,Validation Strategies,Treatment,Source,Library type,Pubmed ID,Public Date,Function/Characteristic of ecDNA,Remarks
0,Homo sapiens,ecDNA,ec_hsa_01,hg19,chr1:148175322-148195932,chr1,148175322.0,148195932.0,Disease,Brain,Glioblastoma,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,WGS,29686388,2018,May confer to divergent inheritance patterns,1.Pathology:GBM;2.This amplicon together with ...
1,Homo sapiens,ecDNA,ec_hsa_02,hg19,chr1:148175386-148295895,chr1,148175386.0,148295895.0,Disease,Brain,Glioblastoma,Prediction,AmpliconArchitect,Not Available,The datasets in form of BAM files from exome s...,WGS,29686388,2018,May confer to divergent inheritance patterns,1.Pathology:GBM;2.This amplicon together with ...
2,Homo sapiens,ecDNA,ec_hsa_03,hg19,chr1:153280226-185860842,chr1,153280226.0,185860842.0,Disease,Brain,Glioblastoma,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,Not Available,29686388,2018,May confer to divergent inheritance patterns,Not Available
3,Homo sapiens,ecDNA,ec_hsa_04,hg19,chr1:154210485-154211045,chr1,154210485.0,154211045.0,Disease,Brain,Glioblastoma,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,WGS,29686388,2018,May confer to divergent inheritance patterns,1.Pathology:GBM;2.This amplicon together with ...
4,Homo sapiens,ecDNA,ec_hsa_05,hg19,chr1:203177968-204957662,chr1,203177968.0,204957662.0,Disease,Brain,Glioblastoma,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,Not Available,29686388,2018,May confer to divergent inheritance patterns,1.This ecDNA is also found in another study(PM...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
637306,Homo sapiens,eccDNA,ecc_hsa_627982,hg19,chrX:53584069-53584485,chrX,53584069.0,53584485.0,Disease,Urine,Chronic Kidney Disease,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...
637307,Homo sapiens,eccDNA,ecc_hsa_627983,hg19,chrX:53584150-53584362,chrX,53584150.0,53584362.0,Disease,Urine,Chronic Kidney Disease,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...
637308,Homo sapiens,eccDNA,ecc_hsa_627984,hg19,chr12:62997317-62997849,chr12,62997317.0,62997849.0,Disease,Urine,Chronic Kidney Disease,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...
637309,Homo sapiens,eccDNA,ecc_hsa_627985,hg19,chr1:26880803-26881180,chr1,26880803.0,26881180.0,Disease,Urine,Chronic Kidney Disease,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...


In [5]:
eccDNA_Atlas["Health/Disease"].value_counts()

Disease    446058
Health     191253
Name: Health/Disease, dtype: int64

In [6]:
eccDNA_Atlas["Disease Name"].value_counts()

Not Available                            191253
Ovarian Cancer                           172771
Prostate Cancer                          141999
Histiocytic Lymphoma                      69420
Cervical Adenocarcinoma                   38876
                                          ...  
Bladder Squamous Cell Carcinoma               1
Renal Cell Carcinoma                          1
Acute Myeloid Leukemia                        1
Pleural Biphasic Mesothelioma                 1
HPV-Mediated Oropharynx Cancer:HPVOPC         1
Name: Disease Name, Length: 111, dtype: int64

In [7]:
eccDNA_Atlas["Genome version"].value_counts()

hg19             637241
Not Available        70
Name: Genome version, dtype: int64

In [76]:
# 假设 eccDNA_Atlas 是你已有的 DataFrame
eccDNA_Atlas_clean = eccDNA_Atlas[
    (eccDNA_Atlas["Health/Disease"] == "Disease") &
    (eccDNA_Atlas["Genome version"] == "hg19")
].copy()

# 应用转换函数
eccDNA_Atlas_clean['ecdna_region_hg38'] = eccDNA_Atlas_clean['Location'].apply(convert_interval)

eccDNA_Atlas_clean

,Species,eccDNA type,eccDNA ID,Genome version,Location,Chr,Start,End,Health/Disease,Tissue/Cell line,...,Experiment/Prediction,Validation Strategies,Treatment,Source,Library type,Pubmed ID,Public Date,Function/Characteristic of ecDNA,Remarks,ecdna_region_hg38
0,Homo sapiens,ecDNA,ec_hsa_01,hg19,chr1:148175322-148195932,chr1,148175322.0,148195932.0,Disease,Brain,...,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,WGS,29686388,2018,May confer to divergent inheritance patterns,1.Pathology:GBM;2.This amplicon together with ...,chr1:145214985_145235595
1,Homo sapiens,ecDNA,ec_hsa_02,hg19,chr1:148175386-148295895,chr1,148175386.0,148295895.0,Disease,Brain,...,Prediction,AmpliconArchitect,Not Available,The datasets in form of BAM files from exome s...,WGS,29686388,2018,May confer to divergent inheritance patterns,1.Pathology:GBM;2.This amplicon together with ...,chr1:148445033_148565542
2,Homo sapiens,ecDNA,ec_hsa_03,hg19,chr1:153280226-185860842,chr1,153280226.0,185860842.0,Disease,Brain,...,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,Not Available,29686388,2018,May confer to divergent inheritance patterns,Not Available,chr1:153307750_185891710
3,Homo sapiens,ecDNA,ec_hsa_04,hg19,chr1:154210485-154211045,chr1,154210485.0,154211045.0,Disease,Brain,...,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,WGS,29686388,2018,May confer to divergent inheritance patterns,1.Pathology:GBM;2.This amplicon together with ...,chr1:154238009_154238569
4,Homo sapiens,ecDNA,ec_hsa_05,hg19,chr1:203177968-204957662,chr1,203177968.0,204957662.0,Disease,Brain,...,Prediction,Manually scrutinized in the Integrative Genomi...,Not Available,The datasets in form of BAM files from exome s...,Not Available,29686388,2018,May confer to divergent inheritance patterns,1.This ecDNA is also found in another study(PM...,chr1:203208840_204988534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
637306,Homo sapiens,eccDNA,ecc_hsa_627982,hg19,chrX:53584069-53584485,chrX,53584069.0,53584485.0,Disease,Urine,...,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...,chrX:53557108_53557524
637307,Homo sapiens,eccDNA,ecc_hsa_627983,hg19,chrX:53584150-53584362,chrX,53584150.0,53584362.0,Disease,Urine,...,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...,chrX:53557189_53557401
637308,Homo sapiens,eccDNA,ecc_hsa_627984,hg19,chr12:62997317-62997849,chr12,62997317.0,62997849.0,Disease,Urine,...,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...,chr12:62603537_62604069
637309,Homo sapiens,eccDNA,ecc_hsa_627985,hg19,chr1:26880803-26881180,chr1,26880803.0,26881180.0,Disease,Urine,...,Prediction,Circle-Seq,Not Available,The data that support the findings of this stu...,Circle-Seq,35474296,2022,1.Ucf-eccDNAs are more frequently derived from...,Urinary cell-free extrachromosomal circular DN...,chr1:26554312_26554689


In [77]:
eccDNA_Atlas["Disease Name"].unique()

array(['Glioblastoma', 'Prostate Cancer', 'Esophageal Cancer',
       'Uterine Corpus Endometrial Cancer', 'Gastric Cancer',
       'Pancreatic Cancer', 'Bladder Cancer', 'Ovarian Cancer',
       'Skin Cancer', 'Lung Adenocarcinoma', 'Breast Cancer',
       'Lung Squamous Cell Cancer', 'Colorectal Cancer',
       'Head and Neck Cancer', 'Uveal Melanoma', 'Sarcoma',
       'Liver Cancer', 'Cervical Cancer', 'Oral Cancer', 'Ewing Sarcoma',
       'Renal Cancer', 'Lower Grade Glioma Cancer',
       'Biliary Tract Cancer', 'Pediatric Brain Cancer',
       'B-cell Lymphoma', 'Colon Cancer', 'Medulloblastoma',
       'Lung Cancer', 'Hematopoietic Cancer', 'Melanoma',
       'Fetal Growth Restrict', 'Not Available',
       'Non-small-cell Lung Cancer', 'Acute Myeloid Leukemia',
       'Myelodysplastic Syndrome', 'Myeloproliferative Neoplasm',
       'Acute Lymphoblastic Leukemia', 'Human KB Epidermoid Carcinoma',
       'Neuroblastoma', 'Acute Kidney Injury', 'Rheumatoid Arthritis',
       'L

In [78]:
# 原始自定义映射（用户已确认）
# 补充的标准化映射（基于已知及广泛使用的缩写）
extended_map = {
    "Prostate Cancer": "PRAD",
    "Esophageal Cancer": "ESCA",
    "Uterine Corpus Endometrial Cancer": "UCEC",
    "Gastric Cancer": "STAD",
    "Bladder Cancer": "BLCA",
    "Ovarian Cancer": "OV",
    "Skin Cancer": "SKCM",
    "Lung Adenocarcinoma": "LUAD",
    "Breast Cancer": "BRCA",
    "Lung Squamous Cell Cancer": "LUSC",
    "Colorectal Cancer": "CRC",
    "Head and Neck Cancer": "HNSCC",
    "Uveal Melanoma": "UVM",
    "Sarcoma": "SARC",
    "Liver Cancer": "LIHC",
    "Cervical Cancer": "CESC",
    "Oral Cancer": "ORAL",
    "Ewing Sarcoma": "EWS",
    "Renal Cancer": "KIRC",
    "Lower Grade Glioma Cancer": "LGG",
    "Biliary Tract Cancer": "CHOL",
    "Pediatric Brain Cancer": "PBCA",
    "B-cell Lymphoma": "BCL",
    "Colon Cancer": "CRC",
    "Medulloblastoma": "MB",
    "Lung Cancer": "NSCLC",
    "Hematopoietic Cancer": "HEMAT",
    "Melanoma": "SKCM",
    "Non-small-cell Lung Cancer": "NSCLC",
    "Acute Myeloid Leukemia": "LAML",
    "Myelodysplastic Syndrome": "MDS",
    "Myeloproliferative Neoplasm": "MPN",
    "Acute Lymphoblastic Leukemia": "ALL",
    "Neuroblastoma": "NB",
    "Leukemia": "LEUK",
    "Gastric Cardia Adenocarcinoma": "STAD",
    "HPV-Mediated Oropharynx Cancer:HPVOPC": "HNSCC",
    "Small Cell Lung Carcinoma": "SCLC",
    "Breast Ductal Carcinoma": "BRCA",
    "Esophageal Squamous Cell Carcinoma": "ESCA",
    "Astrocytoma": "ASTR",
    "Lung Adenosquamous Carcinoma": "LUAS",
    "Gastric Carcinoma": "STAD",
    "Prostate Carcinoma": "PRAD",
    "Endometrial Adenocarcinoma": "UCEC",
    "Gastric Adenosquamous Carcinoma": "STAD",
    "High Grade Ovarian Serous Adenocarcinoma": "OV",
    "Endometrial Adenosquamous Carcinoma": "UCEC",
    "Adult Hepatocellular Carcinoma": "LIHC",
    "Ovarian Clear Cell Adenocarcinoma": "OV",
    "Thyroid Gland Squamous Cell Carcinoma": "THCA",
    "Breast Adenocarcinoma": "BRCA",
    "Bladder Carcinoma": "BLCA",
    "Gastric Tubular Adenocarcinoma": "STAD",
    "Clear Cell Renal Cell Carcinoma": "KIRC",
    "Anaplastic Astrocytoma": "ASTR",
    "Gastric Adenocarcinoma": "STAD",
    "Tongue Squamous Cell Carcinoma": "HNSCC",
    "Acute Monoblastic/Monocytic Leukemia": "LAML",
    "Ovarian Serous Cystadenocarcinoma": "OV",
    "Lung Squamous Cell Carcinoma": "LUSC",
    "Invasive Breast Carcinoma": "BRCA",
    "Cecum Adenocarcinoma": "CRC",
    "Colon Adenocarcinoma": "CRC",
    "Pleural Biphasic Mesothelioma": "MESO",
    "Gliosarcoma": "GBM",
    "Bronchogenic Carcinoma": "LUAD",
    "Breast Carcinoma": "BRCA",
    "Pancreatic Ductal Adenocarcinoma": "PDAC",
    "Lung Large Cell Carcinoma": "LCLC",
    "Cutaneous Melanoma": "SKCM",
    "Esophageal Adenocarcinoma": "ESCA",
    "Colon Carcinoma": "CRC",
    "Anaplastic Large Cell Lymphoma": "ALCL",
    "Invasive Breast Lobular Carcinoma": "BRCA",
    "Multiple Myeloma": "MM",
    "Renal Cell Carcinoma": "KIRC",
    "Bladder Squamous Cell Carcinoma": "BLCA",
    "Ovarian Leiomyosarcoma": "OV",
    "Thyroid Gland Anaplastic Carcinoma": "THCA",
    "Ovarian Mucinous Adenocarcinoma": "OV",
    "Invasive Breast Carcinoma Of No Special Type": "BRCA",
    "Hypopharyngeal Squamous Cell Carcinoma": "HNSCC",
    "Pancreatic Adenocarcinoma": "PDAC",
    "Pancreatic Somatostatinoma": "PDAC",
    "Renal Pelvis Urothelial Carcinoma": "KIRC",
    "Osteosarcoma": "OS",
    "Histiocytic Lymphoma": "HL",
    "Cervical Adenocarcinoma": "CEAD",
    "Ovarian Adenocarcinoma": "OV",
    "Hepatocellular Carcinoma": "LIHC",
    "Malignant Melanoma": "SKCM",
    "Osteogenic Sarcoma": "OS",
    "Leiomyosarcoma": "LMS",
    "Undifferentiated Sarcoma": "SARC",
    "Neurofibroma": "NF",
    "High Grade Serous Ovarian Cancer": "OV",
    "Chronic Kidney Disease": "CKD",
}

# 你可以将该映射与前面的 manual_map 合并以供使用：
#full_map = {**manual_map, **extended_map}

# 应用映射替换
eccDNA_Atlas_clean["sample_type_abbr"] = eccDNA_Atlas_clean["Disease Name"].map(extended_map)

# 若有未覆盖项，保留原始名称以便后续人工审查
#eccDNA_Atlas_clean["sample_type_abbr"] = eccDNA_Atlas_clean["sample_type_abbr"].fillna(eccDNA_Atlas_clean["Disease Name"])

# 查看替换结果的唯一值列表（可用于检查是否仍有未缩写的项目）
print(eccDNA_Atlas_clean["sample_type_abbr"].value_counts(dropna=False))

OV       173341
PRAD     142004
HL        69420
CEAD      38876
HNSCC     10405
NaN        3238
CKD        1758
BRCA       1116
SARC        939
ESCA        734
STAD        544
SKCM        476
LUAD        452
NB          450
BLCA        371
UCEC        270
LUSC        257
SCLC        206
CRC         204
LIHC        183
HEMAT       144
NSCLC       139
LGG         102
KIRC         72
EWS          60
CESC         49
MB           47
LCLC         23
BCL          19
PBCA         17
PDAC         13
ALCL         13
GBM          13
ORAL         10
UVM           8
CHOL          8
THCA          8
ASTR          5
OS            3
LAML          3
MM            2
MESO          1
LUAS          1
Name: sample_type_abbr, dtype: int64


In [79]:
# 显示原始行数
print("原始数据行数：", len(eccDNA_Atlas_clean))

# 统计 NaN 和 Inf 的数量
nan_counts = eccDNA_Atlas_clean[["ecdna_region_hg38"]].isna().sum()
inf_counts = ((eccDNA_Atlas_clean[["ecdna_region_hg38"]] == np.inf) | (eccDNA_Atlas_clean[["ecdna_region_hg38"]] == -np.inf)).sum()

print("NaN counts:\n", nan_counts)
print("Inf counts:\n", inf_counts)

# 清理数据：将 inf 替换为 NaN，然后丢弃包含 NaN 的行
eccDNA_Atlas_clean = eccDNA_Atlas_clean.copy()
eccDNA_Atlas_clean = eccDNA_Atlas_clean.dropna(subset=["ecdna_region_hg38"])

# 显示清理后的行数
print("清理后数据行数：", len(eccDNA_Atlas_clean))

# 预览前几行结果
print(eccDNA_Atlas_clean[["ecdna_region_hg38"]].head())

原始数据行数： 446004
NaN counts:
 ecdna_region_hg38    282
dtype: int64
Inf counts:
 ecdna_region_hg38    0
dtype: int64
清理后数据行数： 445722
          ecdna_region_hg38
0  chr1:145214985_145235595
1  chr1:148445033_148565542
2  chr1:153307750_185891710
3  chr1:154238009_154238569
4  chr1:203208840_204988534


In [82]:
###### database 3：：scEccDNAdb.txt

In [101]:
from pathlib import Path
import pandas as pd
import numpy as np

# 文件路径
file_path = Path(r"E:\05.project\04.ecDNA\01.data\ECDNA-DB\归档\scEccDNAdb.txt")

# 读取 TSV 文件
scEccDNAdb = pd.read_csv(file_path, sep='\t')

# 显示原始行数
print("原始数据行数：", len(scEccDNAdb))

# 统计 NaN 和 Inf 的数量
nan_counts = scEccDNAdb[['ecDNA  Segments']].isna().sum()
inf_counts = ((scEccDNAdb[['ecDNA  Segments']] == np.inf) | (scEccDNAdb[['ecDNA  Segments']] == -np.inf)).sum()

print("NaN counts:\n", nan_counts)
print("Inf counts:\n", inf_counts)

# 清理数据：将 inf 替换为 NaN，然后丢弃包含 NaN 的行
scEccDNAdb_clean = scEccDNAdb.copy()
scEccDNAdb_clean = scEccDNAdb_clean.dropna(subset=['ecDNA  Segments'])

# 显示清理后的行数
print("清理后数据行数：", len(scEccDNAdb_clean))

# 转换为整数类型
# 拼接为 ecdna_region 列
scEccDNAdb_clean["ecdna_region_hg38"] = scEccDNAdb_clean['ecDNA  Segments'].str.replace('-', '_', regex=False)

# 预览前几行结果
print(scEccDNAdb_clean[[ "ecdna_region_hg38"]].head())

原始数据行数： 171
NaN counts:
 ecDNA  Segments    0
dtype: int64
Inf counts:
 ecDNA  Segments    0
dtype: int64
清理后数据行数： 171
           ecdna_region_hg38
0   chr1:190575071_192752074
1  chr11:129468774_129941685
2   chr1:115268880_115508157
3   chr2:187303561_187514897
4   chr9:116067959_116133073


In [103]:
scEccDNAdb_clean["DiseaseType"].value_counts()

Alzheimer's Disease                     49
Rectal Tumor                            29
Gastric Cancer                          18
Breast Cancer                           16
Prostate Cancer                         16
Colon Cancer                            12
Gastroesophageal Junction Cancer         8
Charcot-Marie-Tooth 1A(CMT1A)            7
High-grade Serous Ovarian Cancer         4
Small-Cell Lung Cancer                   3
Cervical Cancer                          3
Non-small Cell Lung Carcinoma(NSCLC)     2
Lung Squamous Cell Carcinoma             2
Urothelial Bladder Carcinoma             1
Leukemia                                 1
Name: DiseaseType, dtype: int64

In [105]:
supplement_map = {
    "Alzheimer's Disease": "ALZ",                        # 神经退行性疾病，非癌症，但用 ALZ 表示
    "Rectal Tumor": "CRC",                             # TCGA: Rectum adenocarcinoma
    "Gastroesophageal Junction Cancer": "STAD",         # 通常归入胃癌 STAD
    "Charcot-Marie-Tooth 1A(CMT1A)": "CMT1A",            # 遗传性神经病，非癌症，但可标记
    "High-grade Serous Ovarian Cancer": "OV",           # 高级别浆液性卵巢癌，属 TCGA OV
    "Small-Cell Lung Cancer": "SCLC",                   # 小细胞肺癌
    "Non-small Cell Lung Carcinoma(NSCLC)": "NSCLC",    # 非小细胞肺癌
    "Urothelial Bladder Carcinoma": "BLCA",             # 膀胱尿路上皮癌
}

# 合并已有映射 + 新补充映射
full_map = {**manual_map, **extended_map, **supplement_map}

# 应用到你的 DataFrame（例如 eccDNA_Atlas、circlebase_clean 等）
scEccDNAdb_clean["sample_type_abbr"] = scEccDNAdb_clean["DiseaseType"].map(full_map)

# 对未匹配的保留原名（便于人工检查）
scEccDNAdb_clean["sample_type_abbr"] = scEccDNAdb_clean["sample_type_abbr"].fillna(scEccDNAdb_clean["DiseaseType"])
scEccDNAdb_clean

,Speices,ecDNA_ID,ecDNA Segments,Size（bp）,Median_feature_CN,Max_feature_CN,Cell Type/Name,Tissue Source,Cell Line Source,DiseaseType,...,Amplicon Intervals,Amplicon OncogenesAmplified,TotalIntervalSize,AmplifiedIntervalSize,AverageAmplifiedCopyCount,BioSampleID,GC%,Remarks,ecdna_region_hg38,sample_type_abbr
0,Homo Sapiens,ecDNA_hsa_01,chr1:190575071-192752074,2177003,4.703301,4.703301,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,chr1:185678402-199712921,"PTPRC,CRB1,CDC73,",14034520,13834508,3.840616,SAMEA13791575,34.708480,NaN,chr1:190575071_192752074,ALZ
1,Homo Sapiens,ecDNA_hsa_02,chr11:129468774-129941685,472911,6.440441,28.108459,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,chr11:128856684-135006516,"PRDM10,",6149833,6024525,8.842979,SAMEA13791631,43.414631,NaN,chr11:129468774_129941685,ALZ
2,Homo Sapiens,ecDNA_hsa_03,chr1:115268880-115508157,239277,13.939340,13.939340,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,chr1:100033884-120573534,"WNT2B,BCL2L15,PSRC1,AHCYL1,ATP1A1,TRIM33,VAV3,...",20539651,20539093,8.107013,SAMEA13791631,37.060657,NaN,chr1:115268880_115508157,ALZ
3,Homo Sapiens,ecDNA_hsa_04,chr2:187303561-187514897,211336,4.537762,4.537762,Neuron,Entorhinal Cortex,NaN,Alzheimer's Disease,...,chr2:178634951-197969529,"PMS1,ASNSD1,",19334579,19333814,2.983216,SAMEA13792219,37.126485,NaN,chr2:187303561_187514897,ALZ
4,Homo Sapiens,ecDNA_hsa_05,chr9:116067959-116133073,65114,116.514030,236.015245,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,"chr7:64971019-152096022,chr9:72798384-141213431","ARHGEF5,KIAA1549,HSPA5,VAV2,TRIM32,CROT,POT1,N...",155540052,154969774,23.712699,SAMEA13791857,44.880596,NaN,chr9:116067959_116133073,ALZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,Homo Sapiens,ecDNA_hsa_167,chr11:2062961-2088894,25933,1173.729380,2399.222126,Single cell of K562,Bone Marrow,K562,Leukemia,...,chr11:1977775-2212779,"H19,",235005,28731,1165.069701,SAMEA13791812,37.352510,NaN,chr11:2062961_2088894,LEUK
167,Homo Sapiens,ecDNA_hsa_168,chr1:103602982-103616662,13680,23.653818,28.006511,Single TOV2295 Cell,Ovary,TOV2295,High-grade Serous Ovarian Cancer,...,chr1:103488902-103738904,NaN,250003,249994,7.096018,SAMEA13791813,33.594036,Ovary Epithelium,chr1:103602982_103616662,OV
168,Homo Sapiens,ecDNA_hsa_169,chr7:47979016-48065343,86327,5.493389,6.908643,Single TOV2295 Cell,Ovary,TOV2295,High-grade Serous Ovarian Cancer,...,chr7:47975719-48065717,NaN,89999,89990,5.720680,SAMEA13791814,41.921509,Ovary Epithelium,chr7:47979016_48065343,OV
169,Homo Sapiens,ecDNA_hsa_170,chr21:35663499-35703278,39779,9.513775,9.513775,Single TOV2295 Cell,Ovary,TOV2295,High-grade Serous Ovarian Cancer,...,chr21:35619936-35784951,NaN,165016,165009,5.529363,SAMEA13791815,42.395676,Ovary Epithelium,chr21:35663499_35703278,OV


In [117]:
# 拆分 "\n" 并展开为多行
scEccDNAdb_clean = scEccDNAdb_clean.assign(
    ecdna_region_hg38=scEccDNAdb_clean["ecdna_region_hg38"].str.split("\n")
).explode("ecdna_region_hg38").reset_index(drop=True)
scEccDNAdb_clean

,Speices,ecDNA_ID,ecDNA Segments,Size（bp）,Median_feature_CN,Max_feature_CN,Cell Type/Name,Tissue Source,Cell Line Source,DiseaseType,...,Amplicon Intervals,Amplicon OncogenesAmplified,TotalIntervalSize,AmplifiedIntervalSize,AverageAmplifiedCopyCount,BioSampleID,GC%,Remarks,ecdna_region_hg38,sample_type_abbr
0,Homo Sapiens,ecDNA_hsa_01,chr1:190575071-192752074,2177003,4.703301,4.703301,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,chr1:185678402-199712921,"PTPRC,CRB1,CDC73,",14034520,13834508,3.840616,SAMEA13791575,34.708480,NaN,chr1:190575071_192752074,ALZ
1,Homo Sapiens,ecDNA_hsa_02,chr11:129468774-129941685,472911,6.440441,28.108459,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,chr11:128856684-135006516,"PRDM10,",6149833,6024525,8.842979,SAMEA13791631,43.414631,NaN,chr11:129468774_129941685,ALZ
2,Homo Sapiens,ecDNA_hsa_03,chr1:115268880-115508157,239277,13.939340,13.939340,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,chr1:100033884-120573534,"WNT2B,BCL2L15,PSRC1,AHCYL1,ATP1A1,TRIM33,VAV3,...",20539651,20539093,8.107013,SAMEA13791631,37.060657,NaN,chr1:115268880_115508157,ALZ
3,Homo Sapiens,ecDNA_hsa_04,chr2:187303561-187514897,211336,4.537762,4.537762,Neuron,Entorhinal Cortex,NaN,Alzheimer's Disease,...,chr2:178634951-197969529,"PMS1,ASNSD1,",19334579,19333814,2.983216,SAMEA13792219,37.126485,NaN,chr2:187303561_187514897,ALZ
4,Homo Sapiens,ecDNA_hsa_05,chr9:116067959-116133073,65114,116.514030,236.015245,Neuron,Temporal Cortex,NaN,Alzheimer's Disease,...,"chr7:64971019-152096022,chr9:72798384-141213431","ARHGEF5,KIAA1549,HSPA5,VAV2,TRIM32,CROT,POT1,N...",155540052,154969774,23.712699,SAMEA13791857,44.880596,NaN,chr9:116067959_116133073,ALZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,Homo Sapiens,ecDNA_hsa_167,chr11:2062961-2088894,25933,1173.729380,2399.222126,Single cell of K562,Bone Marrow,K562,Leukemia,...,chr11:1977775-2212779,"H19,",235005,28731,1165.069701,SAMEA13791812,37.352510,NaN,chr11:2062961_2088894,LEUK
402,Homo Sapiens,ecDNA_hsa_168,chr1:103602982-103616662,13680,23.653818,28.006511,Single TOV2295 Cell,Ovary,TOV2295,High-grade Serous Ovarian Cancer,...,chr1:103488902-103738904,NaN,250003,249994,7.096018,SAMEA13791813,33.594036,Ovary Epithelium,chr1:103602982_103616662,OV
403,Homo Sapiens,ecDNA_hsa_169,chr7:47979016-48065343,86327,5.493389,6.908643,Single TOV2295 Cell,Ovary,TOV2295,High-grade Serous Ovarian Cancer,...,chr7:47975719-48065717,NaN,89999,89990,5.720680,SAMEA13791814,41.921509,Ovary Epithelium,chr7:47979016_48065343,OV
404,Homo Sapiens,ecDNA_hsa_170,chr21:35663499-35703278,39779,9.513775,9.513775,Single TOV2295 Cell,Ovary,TOV2295,High-grade Serous Ovarian Cancer,...,chr21:35619936-35784951,NaN,165016,165009,5.529363,SAMEA13791815,42.395676,Ovary Epithelium,chr21:35663499_35703278,OV


In [ ]:
###### database 4：：eccDNAdb.rds.tsv

In [122]:
from pathlib import Path
import pandas as pd
import numpy as np

# 文件路径
file_path = Path(r"E:\05.project\04.ecDNA\01.data\ECDNA-DB\归档\eccDNAdb.rds.tsv")

# 读取 TSV 文件
eccDNAdb = pd.read_csv(file_path, sep='\t')

# 显示原始行数
print("原始数据行数：", len(eccDNAdb))

# 统计 NaN 和 Inf 的数量
nan_counts = eccDNAdb[['Segments']].isna().sum()
inf_counts = ((eccDNAdb[['Segments']] == np.inf) | (eccDNAdb[['Segments']] == -np.inf)).sum()

print("NaN counts:\n", nan_counts)
print("Inf counts:\n", inf_counts)

# 清理数据：将 inf 替换为 NaN，然后丢弃包含 NaN 的行
eccDNAdb_clean = eccDNAdb.copy()
eccDNAdb_clean = eccDNAdb_clean.dropna(subset=['Segments'])

# 显示清理后的行数
print("清理后数据行数：", len(eccDNAdb_clean))

# 转换为整数类型
# 拼接为 ecdna_region 列
# 去除末尾的 + 或 -
eccDNAdb_clean["Segments"] = eccDNAdb_clean["Segments"].str.rstrip("+-")
eccDNAdb_clean["ecdna_region_hg38"] = eccDNAdb_clean['Segments'].str.replace('-', '_', regex=False)

# 预览前几行结果
print(eccDNAdb_clean[[ "ecdna_region_hg38"]].head())

原始数据行数： 3747
NaN counts:
 Segments    0
dtype: int64
Inf counts:
 Segments    0
dtype: int64
清理后数据行数： 3747
         ecdna_region_hg38
0  chr8:48100000_146364021
1   chr1:86001724_86022428
2     chr6:402729_57295103
3  chr8:48100000_146364021
4  chr11:68225309_71566078


In [124]:
eccDNAdb_clean["Cancer"].value_counts()

Breast cancer                                  434
Sarcoma cancer                                 430
Glioblastoma                                   361
Colon cancer                                   353
Glioma                                         309
Lung cancer (adenocarcinoma)                   264
Gastric cancer                                 206
Esophageal cancer                              196
Skin cancer                                    180
Lung cancer (squamous cell carcinoma)          125
Bladder cancer                                 122
Liver cancer                                    85
Head and neck cancer                            76
Medulloblastoma                                 72
Uterine corpus endometrial cancer               69
Ovary cancer                                    68
Pancreatic cancer                               67
Lower grade glioma cancer                       64
Renal cancer                                    38
Ovary cancer (clear cell carcin

In [128]:
additional_map = {
    "Breast cancer": "BRCA",
    "Sarcoma cancer": "SARC",
    "Glioblastoma": "GBM",
    "Colon cancer": "CRC",  # 可与 Rectal 统一归为 CRC
    "Glioma": "GLIOMA",     # 泛指，可和 LGG/GBM 区分；若不细分，保留为 GLIOMA
    "Lung cancer (adenocarcinoma)": "LUAD",
    "Gastric cancer": "STAD",
    "Esophageal cancer": "ESCA",
    "Skin cancer": "SKCM",  # TCGA: Skin Cutaneous Melanoma
    "Lung cancer (squamous cell carcinoma)": "LUSC",
    "Bladder cancer": "BLCA",
    "Liver cancer": "LIHC",
    "Head and neck cancer": "HNSCC",
    "Medulloblastoma": "MB",  # 非 TCGA，常用缩写
    "Uterine corpus endometrial cancer": "UCEC",
    "Ovary cancer": "OV",
    "Pancreatic cancer": "PAAD",
    "Lower grade glioma cancer": "LGG",
    "Renal cancer": "KIRC",  # 通常指 clear cell
    "Ovary cancer (clear cell carcinoma)": "OV",  # 子类型，保留为特殊名
    "Ewing sarcoma cancer": "ES",  # 非 TCGA，常用缩写
    "Prostate cancer": "PRAD",
    "Cervical cancer": "CESC",
    "Colorectal cancer": "CRC",
    "Breast cancer (ductal carcinoma)": "BRCA",  # 子类型，可保留原始子名
    "Pediatric brain cancer": "PBC",  # 无标准缩写，保留别名
    "Prostate cancer (adenocarcinoma)": "PRAD",
    "Acute myeloid leukemia": "LAML",
    "Pancreatic cancer (ductal adenocarcinoma)": "PAAD",
    "Chronic myeloid leukaemia": "CML",  # 非 TCGA
    "Hepatocellular carcinoma": "LIHC",
    "Lung cancer/BM": "LUAD",  # 推测为转移瘤，保留原名
    "Ovary cancer (adenocarcinoma)": "OV",
    "Uveal melanoma": "UVM",
    "Lung cancer (large cell carcinoma)": "LCLC",  # 常用缩写
    "B-cell lymphoma": "BCL",
    "Pancreatic cancer (adenosquamous carcinoma)": "PDAC",
    "Thyroid cancer": "THCA",
    "Renal cancer (renal clear cell carcinoma)": "KIRC",
    "Colon cancer/BM": "CRC",
    "Oral cancer": "HNSCC",  # 属 HNSCC，细分口腔
    "Biliary tract cancer": "BTC",  # 胆道癌，常用 BTC 表示
}

# 合并所有映射
full_map = {**manual_map, **extended_map, **supplement_map, **additional_map}

# 应用于 scEccDNAdb_clean 或其他 DataFrame
eccDNAdb_clean["sample_type_abbr"] = eccDNAdb_clean["Cancer"].map(full_map)

# 对未映射项保留原始名称以便检查
eccDNAdb_clean["sample_type_abbr"] = eccDNAdb_clean["sample_type_abbr"].fillna(eccDNAdb_clean["Cancer"])
eccDNAdb_clean

,Run,Number,eccDNAdb_ID,Segments,Copy_count,Sample.source,Sample.type,Tissue,Cancer,Disease,Cell.line,ecdna_region_hg38,sample_type_abbr
0,10.1038/s41588-020-0678-2--SA270384,858,hsa_Chr8_1S_81,chr8:48100000-146364021,-1.000000,10.1038/s41588-020-0678-2,TP,Liver,Liver cancer,Liver cancer,NaN,chr8:48100000_146364021,LIHC
1,10.1038/s41588-020-0678-2--SA270396,789,hsa_3Chr_3S_3,chr1:86001724-86022428,-1.000000,10.1038/s41588-020-0678-2,TP,Liver,Liver cancer,Liver cancer,NaN,chr1:86001724_86022428,LIHC
2,10.1038/s41588-020-0678-2--SA270396,789,hsa_3Chr_3S_3,chr6:402729-57295103,-1.000000,10.1038/s41588-020-0678-2,TP,Liver,Liver cancer,Liver cancer,NaN,chr6:402729_57295103,LIHC
3,10.1038/s41588-020-0678-2--SA270396,789,hsa_3Chr_3S_3,chr8:48100000-146364021,-1.000000,10.1038/s41588-020-0678-2,TP,Liver,Liver cancer,Liver cancer,NaN,chr8:48100000_146364021,LIHC
4,10.1038/s41588-020-0678-2--SA270416,856,hsa_Chr11_3S_2,chr11:68225309-71566078,-1.000000,10.1038/s41588-020-0678-2,TP,Liver,Liver cancer,Liver cancer,NaN,chr11:68225309_71566078,LIHC
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742,SRR964238,619,hsa_Chr7_8S_2,chr7:54521572-54522360,1.010753,SRA,Unspecified,Unspecified,Glioma,Glioma,NaN,chr7:54521572_54522360,GLIOMA
3743,SRR964238,620,hsa_Chr7_4S_7,chr7:54749086-55066284,1.009008,SRA,Unspecified,Unspecified,Glioma,Glioma,NaN,chr7:54749086_55066284,GLIOMA
3744,SRR964238,620,hsa_Chr7_4S_7,chr7:55125071-55125525,1.009008,SRA,Unspecified,Unspecified,Glioma,Glioma,NaN,chr7:55125071_55125525,GLIOMA
3745,SRR964238,620,hsa_Chr7_4S_7,chr7:55207418-55241473,1.009008,SRA,Unspecified,Unspecified,Glioma,Glioma,NaN,chr7:55207418_55241473,GLIOMA


In [ ]:
######  inferecc  数据

In [49]:
spe_com_ecdna = pd.read_csv('./fig_100k/cell_ratio/common/f09-ecdna_cancer_common-cell_ratio_clustermap_by-sample_xy_common-specific_all-ecdna.tsv', sep="\t",index_col=0)
spe_com_ecdna

# ---------- Step 1：复制行名到新列 ----------
spe_com_ecdna["ecdna_cpr"] = spe_com_ecdna.index

# ---------- Step 2：筛选 label_abbr == "specific_positive" ----------
spe_ecdna_df = spe_com_ecdna[spe_com_ecdna["label_abbr"] == "specific_positive"].copy()
spe_ecdna_df

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,label,label_abbr,ecdna_cpr
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr15:50100000_50200000
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr2:176700000_176800000
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:169400000_169500000
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr10:81000000_81100000
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:98800000_98900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,UCEC_specific_positive,specific_positive,chrX:120300000_120400000
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,UCEC_specific_positive,specific_positive,chr5:114000000_114100000
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,UCEC_specific_positive,specific_positive,chr1:214800000_214900000
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,UCEC_specific_positive,specific_positive,chr5:35700000_35800000


In [17]:
spe_com_ecdna["label_abbr"].value_counts()

specific_positive    4973
specific_negative    1848
common               1739
Name: label_abbr, dtype: int64

In [44]:
### 提示词
"""
1. 对 df的 label 这一列的每一个元素进行字符串分割，关键字为下划线，去分割后的第一部分，得到新的一列，新的一列列名为 cancer_type。
2. 对df的每一行依次执行下述操作：

1. 提取 df 的 ecdna_cpr 这一列的元素，记为 ecdna_cpr0，同时提取这一行的 cancer_type，记为cancer_type0, 
2. 在circlebase_clean 表格里的  sample_type_abbr  这一列元素进行查询：得到与记为cancer_type0元素相同的行，提取对应行的ecdna_region_hg38，得到集合S1：
3. 比较 ecdna_cpr0 和 S1的每一个元素（每个元素是一个region ），根据以下has_overlap函数，比较 S1 中有哪些元素  与  ecdna_cpr0 有交集，得到有交集的S1元素，一个新的子集：S2。
4. 在 df 中得到  ecdna_cpr0 元素的所在行的新的一列的对应的元素值 即：S2，该列列名为  circlebase_match。若S2为空，则对应的 circlebase_match  元素值 记为空值。
(需要对has_overlap函数进行一定优化，判定交集的条件适当放松，两个区域之间的距离在50000以内，都算作交集。所以要给has_overlap函数多添加一个参数，
 即相交的距离，设置为传参，本次使用50000)
 """

'\n1. spe_com_ecdna df 添加一列，复制行名，得到新的一列，列名为：ecdnacpr 。对新的df，筛选  label_abbr  这一列为  specific_positive  的行。\n2. 对 新的df的   label  这一列的每一个元素进行字符串分割，关键字为下划线，去分割后的第一部分，得到新的一列，新的一列列名为cancertype。\n3. 按照  label 的这一列的元素的 unique 的值的情况，提取 df的ecdnacpr这一列的元素，得到每一种的 label 的 ecdnacpr，记为集合S0。\n4. 对cancertype 在 circlebase_clean 表格里的  sample_type_abbr  这一列进行查询，得到数据库的对应类型的  cancer  的  ecdna_region_hg38  这一列元素的集合，记为S1\n5. 比较 S0的每一个元素 和 S1的每一个元素，每个元素是一个region ，根据以下has_overlap函数，计算S0中有多少元素  与 S1的有交集。\n6. (需要对has_overlap函数进行一定优化，判定交集的条件适当放松，两个区域之间的距离在50000以内，都算作交集。所以要给has_overlap函数多添加一个参数，\n 即相交的距离，设置为传参，本次使用50000)\n7. 输出S0的交集元素，以及对应的S1的交集元素。\n'

In [ ]:
###### 计算交集

In [70]:
### 1. circlebase_clean

import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean

ref_db = circlebase_clean.copy()

# 复制索引为新列
spe_ecdna_df["ecdna_cpr"] = spe_ecdna_df.index

# 提取 cancer_type（label 以"_"分割，取第一部分）
spe_ecdna_df["cancer_type"] = spe_ecdna_df["label"].str.split("_").str[0]

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in spe_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
spe_ecdna_df["circlebase_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
print(spe_ecdna_df[["ecdna_cpr", "cancer_type", "circlebase_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
spe_ecdna_df["has_match"] = spe_ecdna_df["circlebase_match"].notna()

# 按 cancer_type 分组，统计匹配比例
match_stats = spe_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
match_stats["match_ratio"] = match_stats["match_count"] / match_stats["total_count"]

# 查看统计结果
print(match_stats)

spe_ecdna_df

                                         ecdna_cpr cancer_type  \
chr15:50100000_50200000    chr15:50100000_50200000        BRCA   
chr2:176700000_176800000  chr2:176700000_176800000        BRCA   
chr4:169400000_169500000  chr4:169400000_169500000        BRCA   
chr10:81000000_81100000    chr10:81000000_81100000        BRCA   
chr4:98800000_98900000      chr4:98800000_98900000        BRCA   

                                   circlebase_match  
chr15:50100000_50200000   [chr15:45778668_52964692]  
chr2:176700000_176800000                        NaN  
chr4:169400000_169500000                        NaN  
chr10:81000000_81100000                         NaN  
chr4:98800000_98900000                          NaN  
             total_count  match_count  match_ratio
cancer_type                                       
BRCA                4098         1306     0.318692
CESC                   7            2     0.285714
CRC                   29            3     0.103448
HNSCC                  8

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,label,label_abbr,ecdna_cpr,cancer_type,circlebase_match,has_match
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],True
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr2:176700000_176800000,BRCA,NaN,False
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:169400000_169500000,BRCA,NaN,False
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr10:81000000_81100000,BRCA,NaN,False
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:98800000_98900000,BRCA,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,UCEC_specific_positive,specific_positive,chrX:120300000_120400000,UCEC,NaN,False
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,UCEC_specific_positive,specific_positive,chr5:114000000_114100000,UCEC,NaN,False
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,UCEC_specific_positive,specific_positive,chr1:214800000_214900000,UCEC,NaN,False
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,UCEC_specific_positive,specific_positive,chr5:35700000_35800000,UCEC,[chr5:32261453_41811742],True


In [81]:
### 2. eccDNA_Atlas_clean

import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean

ref_db = eccDNA_Atlas_clean.copy()

# 复制索引为新列
spe_ecdna_df["ecdna_cpr"] = spe_ecdna_df.index

# 提取 cancer_type（label 以"_"分割，取第一部分）
spe_ecdna_df["cancer_type"] = spe_ecdna_df["label"].str.split("_").str[0]

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in spe_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
spe_ecdna_df["eccDNA_Atlas_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
print(spe_ecdna_df[["ecdna_cpr", "cancer_type", "eccDNA_Atlas_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
spe_ecdna_df["has_match"] = spe_ecdna_df["eccDNA_Atlas_match"].notna()

# 按 cancer_type 分组，统计匹配比例
match_stats = spe_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
match_stats["match_ratio"] = match_stats["match_count"] / match_stats["total_count"]

# 查看统计结果
print(match_stats)

spe_ecdna_df

                                         ecdna_cpr cancer_type  \
chr15:50100000_50200000    chr15:50100000_50200000        BRCA   
chr2:176700000_176800000  chr2:176700000_176800000        BRCA   
chr4:169400000_169500000  chr4:169400000_169500000        BRCA   
chr10:81000000_81100000    chr10:81000000_81100000        BRCA   
chr4:98800000_98900000      chr4:98800000_98900000        BRCA   

                                 eccDNA_Atlas_match  
chr15:50100000_50200000   [chr15:45778668_52964692]  
chr2:176700000_176800000                        NaN  
chr4:169400000_169500000                        NaN  
chr10:81000000_81100000                         NaN  
chr4:98800000_98900000                          NaN  
             total_count  match_count  match_ratio
cancer_type                                       
BRCA                4098         1492     0.364080
CESC                   7            2     0.285714
CRC                   29            6     0.206897
HNSCC                  8

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,label,label_abbr,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],True,[chr15:45778668_52964692]
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr2:176700000_176800000,BRCA,NaN,False,NaN
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:169400000_169500000,BRCA,NaN,False,NaN
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr10:81000000_81100000,BRCA,NaN,False,NaN
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:98800000_98900000,BRCA,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,UCEC_specific_positive,specific_positive,chrX:120300000_120400000,UCEC,NaN,False,NaN
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,UCEC_specific_positive,specific_positive,chr5:114000000_114100000,UCEC,NaN,False,NaN
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,UCEC_specific_positive,specific_positive,chr1:214800000_214900000,UCEC,NaN,False,NaN
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,UCEC_specific_positive,specific_positive,chr5:35700000_35800000,UCEC,[chr5:32261453_41811742],True,[chr5:32261453_41811742]


In [119]:
###### 3. scEccDNAdb_clean

import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean

ref_db = scEccDNAdb_clean.copy()

# 复制索引为新列
spe_ecdna_df["ecdna_cpr"] = spe_ecdna_df.index

# 提取 cancer_type（label 以"_"分割，取第一部分）
spe_ecdna_df["cancer_type"] = spe_ecdna_df["label"].str.split("_").str[0]

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in spe_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
spe_ecdna_df["scEccDNAdb_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
print(spe_ecdna_df[["ecdna_cpr", "cancer_type", "scEccDNAdb_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
spe_ecdna_df["has_match"] = spe_ecdna_df["scEccDNAdb_match"].notna()

# 按 cancer_type 分组，统计匹配比例
match_stats = spe_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
match_stats["match_ratio"] = match_stats["match_count"] / match_stats["total_count"]

# 查看统计结果
print(match_stats)

spe_ecdna_df

                                         ecdna_cpr cancer_type  \
chr15:50100000_50200000    chr15:50100000_50200000        BRCA   
chr2:176700000_176800000  chr2:176700000_176800000        BRCA   
chr4:169400000_169500000  chr4:169400000_169500000        BRCA   
chr10:81000000_81100000    chr10:81000000_81100000        BRCA   
chr4:98800000_98900000      chr4:98800000_98900000        BRCA   

                         scEccDNAdb_match  
chr15:50100000_50200000               NaN  
chr2:176700000_176800000              NaN  
chr4:169400000_169500000              NaN  
chr10:81000000_81100000               NaN  
chr4:98800000_98900000                NaN  
             total_count  match_count  match_ratio
cancer_type                                       
BRCA                4098           57     0.013909
CESC                   7            0     0.000000
CRC                   29            0     0.000000
HNSCC                  8            0     0.000000
MM                   362         

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,label,label_abbr,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match,scEccDNAdb_match
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],False,[chr15:45778668_52964692],NaN
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr2:176700000_176800000,BRCA,NaN,False,NaN,NaN
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:169400000_169500000,BRCA,NaN,False,NaN,NaN
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr10:81000000_81100000,BRCA,NaN,False,NaN,NaN
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:98800000_98900000,BRCA,NaN,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,UCEC_specific_positive,specific_positive,chrX:120300000_120400000,UCEC,NaN,False,NaN,NaN
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,UCEC_specific_positive,specific_positive,chr5:114000000_114100000,UCEC,NaN,False,NaN,NaN
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,UCEC_specific_positive,specific_positive,chr1:214800000_214900000,UCEC,NaN,False,NaN,NaN
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,UCEC_specific_positive,specific_positive,chr5:35700000_35800000,UCEC,[chr5:32261453_41811742],False,[chr5:32261453_41811742],NaN


In [129]:
##### 4.eccDNAdb_clean

import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean

ref_db = eccDNAdb_clean.copy()

# 复制索引为新列
spe_ecdna_df["ecdna_cpr"] = spe_ecdna_df.index

# 提取 cancer_type（label 以"_"分割，取第一部分）
spe_ecdna_df["cancer_type"] = spe_ecdna_df["label"].str.split("_").str[0]

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in spe_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
spe_ecdna_df["eccDNAdb_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
print(spe_ecdna_df[["ecdna_cpr", "cancer_type", "eccDNAdb_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
spe_ecdna_df["has_match"] = spe_ecdna_df["eccDNAdb_match"].notna()

# 按 cancer_type 分组，统计匹配比例
match_stats = spe_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
match_stats["match_ratio"] = match_stats["match_count"] / match_stats["total_count"]

# 查看统计结果
print(match_stats)

spe_ecdna_df

                                         ecdna_cpr cancer_type eccDNAdb_match
chr15:50100000_50200000    chr15:50100000_50200000        BRCA            NaN
chr2:176700000_176800000  chr2:176700000_176800000        BRCA            NaN
chr4:169400000_169500000  chr4:169400000_169500000        BRCA            NaN
chr10:81000000_81100000    chr10:81000000_81100000        BRCA            NaN
chr4:98800000_98900000      chr4:98800000_98900000        BRCA            NaN
             total_count  match_count  match_ratio
cancer_type                                       
BRCA                4098          636     0.155198
CESC                   7            2     0.285714
CRC                   29            2     0.068966
HNSCC                  8            2     0.250000
MM                   362            0     0.000000
OV                    38           13     0.342105
PDAC                  52            0     0.000000
SKCM                  71           15     0.211268
UCEC                 3

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,label,label_abbr,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],False,[chr15:45778668_52964692],NaN,NaN
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr2:176700000_176800000,BRCA,NaN,False,NaN,NaN,NaN
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:169400000_169500000,BRCA,NaN,False,NaN,NaN,NaN
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr10:81000000_81100000,BRCA,NaN,False,NaN,NaN,NaN
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,BRCA_specific_positive,specific_positive,chr4:98800000_98900000,BRCA,NaN,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,UCEC_specific_positive,specific_positive,chrX:120300000_120400000,UCEC,NaN,False,NaN,NaN,NaN
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,UCEC_specific_positive,specific_positive,chr5:114000000_114100000,UCEC,NaN,False,NaN,NaN,NaN
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,UCEC_specific_positive,specific_positive,chr1:214800000_214900000,UCEC,NaN,False,NaN,NaN,NaN
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,UCEC_specific_positive,specific_positive,chr5:35700000_35800000,UCEC,[chr5:32261453_41811742],False,[chr5:32261453_41811742],NaN,NaN


In [ ]:
讨论：
1.展示总体相似性
2.MITF  在数据库里是不是也是specific的？
3.common ？怎么统计 数据库的 common 的问题？1.严格统计 common 2.宽松统计
4.
（1）实验验证 真实的ecDNA 相对可靠 
（2）SKCM MITF，specific，可以作为其他的specific ecDNA的支撑（强调我们自己发现的肿瘤特异性）
（3）实验 印证

1. FISH match 对应到 CRC SKCM OV BRCA；FISH 作为支持 specific 支持结果
2. 反向统计 更有意义

In [163]:
eccDNA_Atlas_clean["Validation Strategies"].value_counts()

By paired-end sequencing                                                                         312406
By paired-end sequencing of MDA products                                                         108295
Circle-Seq                                                                                        13732
AmpliconArchitect                                                                                  7424
FISH&ATAC-seq                                                                                      1782
FISH&AmpliconArchitect                                                                              468
Scanning Electron Microscopy &Transmission Electron Microscopy &Circle-Seq                          464
Short-read sequencing&Nanopore sequencing                                                           418
FISH&(Manually scrutinized in the Integrative Genomics Viewer/Predicted by AmpliconArchitect)       223
Manually scrutinized in the Integrative Genomics Viewer/Predicte

In [164]:
# 筛选以 "FISH" 开头的行
fish_df = eccDNA_Atlas_clean[eccDNA_Atlas_clean["Validation Strategies"].str.startswith("FISH")]
fish_df["Validation Strategies"].value_counts()

FISH&ATAC-seq                                                                                    1782
FISH&AmpliconArchitect                                                                            468
FISH&(Manually scrutinized in the Integrative Genomics Viewer/Predicted by AmpliconArchitect)     223
FISH&Using short read data                                                                         78
FISH&Amplicon reconstruction                                                                       29
FISH&Quantitative PCR&Chromosome Walking                                                           21
FISH&High-resolution array CGH                                                                      9
FISH&Reconstructed by WGS                                                                           2
Name: Validation Strategies, dtype: int64

In [199]:
###### database 5：：fish_df

In [201]:
fish_df["sample_type_abbr"].value_counts()

PRAD     1293
OV        576
BRCA       79
CRC        73
NSCLC      52
NB         29
MB         24
HEMAT      16
SKCM       15
KIRC        6
Name: sample_type_abbr, dtype: int64

In [165]:
import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean

ref_db = fish_df.copy()

# 复制索引为新列
spe_ecdna_df["ecdna_cpr"] = spe_ecdna_df.index

# 提取 cancer_type（label 以"_"分割，取第一部分）
spe_ecdna_df["cancer_type"] = spe_ecdna_df["label"].str.split("_").str[0]

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in spe_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
spe_ecdna_df["FISH_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
print(spe_ecdna_df[["ecdna_cpr", "cancer_type", "FISH_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
spe_ecdna_df["has_match"] = spe_ecdna_df["FISH_match"].notna()

# 按 cancer_type 分组，统计匹配比例
match_stats = spe_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
match_stats["match_ratio"] = match_stats["match_count"] / match_stats["total_count"]

# 查看统计结果
print(match_stats)

spe_ecdna_df

                                         ecdna_cpr cancer_type FISH_match
chr15:50100000_50200000    chr15:50100000_50200000        BRCA        NaN
chr2:176700000_176800000  chr2:176700000_176800000        BRCA        NaN
chr4:169400000_169500000  chr4:169400000_169500000        BRCA        NaN
chr10:81000000_81100000    chr10:81000000_81100000        BRCA        NaN
chr4:98800000_98900000      chr4:98800000_98900000        BRCA        NaN
             total_count  match_count  match_ratio
cancer_type                                       
BRCA                4098          334     0.081503
CESC                   7            0     0.000000
CRC                   29            1     0.034483
HNSCC                  8            0     0.000000
MM                   362            0     0.000000
OV                    38           20     0.526316
PDAC                  52            0     0.000000
SKCM                  71            3     0.042254
UCEC                 308            0     0.00

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,...,label_abbr,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match,all_match,FISH_match
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,specific_positive,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],False,[chr15:45778668_52964692],NaN,NaN,True,NaN
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,specific_positive,chr2:176700000_176800000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,specific_positive,chr4:169400000_169500000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,specific_positive,chr10:81000000_81100000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,specific_positive,chr4:98800000_98900000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,...,specific_positive,chrX:120300000_120400000,UCEC,NaN,False,NaN,NaN,NaN,False,NaN
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,...,specific_positive,chr5:114000000_114100000,UCEC,NaN,False,NaN,NaN,NaN,False,NaN
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,...,specific_positive,chr1:214800000_214900000,UCEC,NaN,False,NaN,NaN,NaN,False,NaN
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,...,specific_positive,chr5:35700000_35800000,UCEC,[chr5:32261453_41811742],False,[chr5:32261453_41811742],NaN,NaN,True,NaN


In [180]:
spe_ecdna_df["fish_match_tf"] = spe_ecdna_df[["FISH_match"]].notna().any(axis=1)
spe_ecdna_df_fish = spe_ecdna_df[spe_ecdna_df["fish_match_tf"]]
spe_ecdna_df_fish

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,...,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match,all_match,FISH_match,fish_match_tf
chr11:30200000_30300000,0.000088,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,chr11:30200000_30300000,BRCA,"[chr11:29903454_37172450, chr11:29437670_30406...",True,"[chr11:29437670_30406715, chr11:30195955_41827...",NaN,[chr11:30217502_41849393],True,[chr11:29903454_37172450],True
chr16:53300000_53400000,0.000088,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,chr16:53300000_53400000,BRCA,[chr16:46502089_54801288],True,"[chr16:46502089_54801288, chr16:46532089_57285...",NaN,NaN,True,[chr16:46502089_54801288],True
chr17:40700000_40800000,0.000215,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,chr17:40700000_40800000,BRCA,"[chr17:38519765_41407748, chr17:38355421_42799...",True,"[chr17:38355421_42799070, chr17:38597293_41070...",NaN,[chr17:36511304_40951088],True,[chr17:38519765_41407748],True
chr17:54800000_54900000,0.001369,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,chr17:54800000_54900000,BRCA,"[chr17:47486635_65386882, chr17:49794933_71184...",True,"[chr17:49794933_71184862, chr17:49931174_61875...",NaN,"[chr17:53285381_65824101, chr17:47872295_69181...",True,"[chr17:53488240_76801518, chr17:48083839_65651...",True
chr8:131000000_131100000,0.000205,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,chr8:131000000_131100000,BRCA,"[chr8:100167793_141382963, chr8:103173427_1450...",True,"[chr8:100167793_141382963, chr8:103173427_1453...",NaN,"[chr8:104185655_146364022, chr8:124808291_1310...",True,[chr8:77966766_145451787],True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chr3:111000000_111100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000963,0.0,0.000000,0.0,...,chr3:111000000_111100000,OV,[chr3:110899132_121011166],True,"[chr3:110899132_121011166, chr3:110958942_1109...",NaN,"[chr3:93900000_198022429, chr3:94612237_198022...",True,[chr3:107765601_139126312],True
chr6:75300000_75400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000321,0.0,0.000000,0.0,...,chr6:75300000_75400000,OV,[chr6:71110281_76361657],True,"[chr6:71110281_76361657, chr6:75262060_7526235...",NaN,NaN,True,[chr6:73506891_107347485],True
chr3:59200000_59300000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000493,0.0,...,chr3:59200000_59300000,SKCM,NaN,True,"[chr3:54283474_75618349, chr3:54283474_75628349]",NaN,NaN,True,"[chr3:54283474_75618349, chr3:54283474_75628349]",True
chr3:69900000_70000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.005582,0.0,...,chr3:69900000_70000000,SKCM,"[chr3:63217388_87850849, chr3:67627380_7641020...",True,"[chr3:63217388_87850849, chr3:67627380_7641020...",NaN,"[chr3:63203064_87899999, chr3:67677804_76459356]",True,"[chr3:54283474_75618349, chr3:54283474_75628349]",True


In [181]:
spe_ecdna_df_fish["chr_raw"] = spe_ecdna_df_fish["ecdna_cpr"]

df_ecdna = spe_ecdna_df_fish.copy()
df_ecdna_gene = genebody_region(df_fragments=df_ecdna,species="hg38")
df_ecdna_gene['oncogene'] = df_ecdna_gene['genebody_region_gene'].apply(custom_transform)

species value: hg38


In [185]:
df_ecdna_gene

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,...,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match,all_match,FISH_match,fish_match_tf,chr_raw,genebody_region,genebody_region_gene,oncogene
chr11:30200000_30300000,0.000088,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,"[chr11:29437670_30406715, chr11:30195955_41827...",NaN,[chr11:30217502_41849393],True,[chr11:29903454_37172450],True,chr11:30200000_30300000,1,[FSHB],[]
chr16:53300000_53400000,0.000088,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,"[chr16:46502089_54801288, chr16:46532089_57285...",NaN,NaN,True,[chr16:46502089_54801288],True,chr16:53300000_53400000,9,"[CHD9, RP11-454F8.4, RNA5SP427, RP11-44F14.4, ...",[]
chr17:40700000_40800000,0.000215,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,"[chr17:38355421_42799070, chr17:38597293_41070...",NaN,[chr17:36511304_40951088],True,[chr17:38519765_41407748],True,chr17:40700000_40800000,6,"[KRT24, KRT223P, KRT25, KRT26, KRT27, KRT28]",[]
chr17:54800000_54900000,0.001369,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,"[chr17:49794933_71184862, chr17:49931174_61875...",NaN,"[chr17:53285381_65824101, chr17:47872295_69181...",True,"[chr17:53488240_76801518, chr17:48083839_65651...",True,chr17:54800000_54900000,1,[TOM1L1],[]
chr8:131000000_131100000,0.000205,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,"[chr8:100167793_141382963, chr8:103173427_1453...",NaN,"[chr8:104185655_146364022, chr8:124808291_1310...",True,[chr8:77966766_145451787],True,chr8:131000000_131100000,1,[ADCY8],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chr3:111000000_111100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000963,0.0,0.000000,0.0,...,"[chr3:110899132_121011166, chr3:110958942_1109...",NaN,"[chr3:93900000_198022429, chr3:94612237_198022...",True,[chr3:107765601_139126312],True,chr3:111000000_111100000,2,"[PVRL3-AS1, NECTIN3]",[]
chr6:75300000_75400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000321,0.0,0.000000,0.0,...,"[chr6:71110281_76361657, chr6:75262060_7526235...",NaN,NaN,True,[chr6:73506891_107347485],True,chr6:75300000_75400000,3,"[FILIP1, HMGB1P39, RP11-415D17.3]",[]
chr3:59200000_59300000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000493,0.0,...,"[chr3:54283474_75618349, chr3:54283474_75628349]",NaN,NaN,True,"[chr3:54283474_75618349, chr3:54283474_75628349]",True,chr3:59200000_59300000,0,0,0
chr3:69900000_70000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.005582,0.0,...,"[chr3:63217388_87850849, chr3:67627380_7641020...",NaN,"[chr3:63203064_87899999, chr3:67677804_76459356]",True,"[chr3:54283474_75618349, chr3:54283474_75628349]",True,chr3:69900000_70000000,2,"[MITF, SAMMSON]",[MITF]


In [189]:
df_merge_onco = group_merge(df_ecdna_gene,"cancer_type","oncogene")
# 计算col1列每个元素的长度，并创建新的col2列
df_merge_onco['oncogene_nor_number'] = df_merge_onco['oncogene_nor'].apply(lambda x: len(x))
df_merge_onco

,cancer_type,oncogene_nor,most_oncogene_nor,oncogene_nor_number
0,BRCA,"[FAM135B, PTPRT, BRIP1, EIF3E, PTPRT, BMP5, PT...","[PTPRT, CSMD3, EIF3E, CNBD1, FAM135B, BRIP1, B...",19
1,CRC,[],[],0
2,OV,[],[],0
3,SKCM,[MITF],[MITF],1


In [191]:
print(df_merge_onco["most_oncogene_nor"][0])

['PTPRT', 'CSMD3', 'EIF3E', 'CNBD1', 'FAM135B', 'BRIP1', 'BMP5', 'RSPO2', 'RUNX1T1', 'HLF']


In [186]:
df_merge_body = group_merge(df_ecdna_gene,"cancer_type","genebody_region_gene")
# 计算col1列每个元素的长度，并创建新的col2列
df_merge_body['genebody_region_gene_nor_number'] = df_merge_body['genebody_region_gene_nor'].apply(lambda x: len(x))
df_merge_body

,cancer_type,genebody_region_gene_nor,most_genebody_region_gene_nor,genebody_region_gene_nor_number
0,BRCA,"[FSHB, CHD9, RP11-454F8.4, RNA5SP427, RP11-44F...","[ZFPM2, RIMS2, DCDC1, PTPRT, RALYL, CSMD3, MMP...",578
1,CRC,[CDH13],[CDH13],1
2,OV,"[RP11-10O22.1, LINC01192, BDNF-AS, LINC00678, ...","[RP11-10O22.1, LINC01192, BDNF-AS, LINC00678, ...",40
3,SKCM,"[MITF, SAMMSON, RP11-24H1.1, LINC00698]","[MITF, SAMMSON, RP11-24H1.1, LINC00698]",4


In [ ]:
##### 总计

In [166]:
# Step 1: 基础统计（每种 cancer_type 的总样本数）
match_stats = spe_ecdna_df.groupby("cancer_type").size().reset_index(name="total_count")

# Step 2: 定义四个数据库的匹配列
match_columns = [
    "circlebase_match",
    "eccDNA_Atlas_match",
    "eccDNAdb_match",
    "scEccDNAdb_match",
    "FISH_match"
]

# Step 3: 分别统计四个数据库的匹配 count 和 ratio
for col in match_columns:
    tmp = spe_ecdna_df.copy()
    tmp["has_match"] = tmp[col].notna()
    
    stats = (
        tmp.groupby("cancer_type")["has_match"]
        .agg(match_count="sum")
        .reset_index()
        .rename(columns={"match_count": f"{col}_count"})
    )
    
    match_stats = match_stats.merge(stats, on="cancer_type", how="left")

# Step 4: 计算匹配比例
for col in match_columns:
    match_stats[f"{col}_ratio"] = match_stats[f"{col}_count"] / match_stats["total_count"]

# Step 5: 整合“是否任一数据库有匹配”
spe_ecdna_df["all_match"] = spe_ecdna_df[match_columns].notna().any(axis=1)

# Step 6: 统计整合后的匹配情况
all_match_stats = (
    spe_ecdna_df.groupby("cancer_type")["all_match"]
    .agg(all_match_count="sum")
    .reset_index()
)
match_stats = match_stats.merge(all_match_stats, on="cancer_type", how="left")
match_stats["all_match_ratio"] = match_stats["all_match_count"] / match_stats["total_count"]

# Step 7: 排序 & 展示
match_stats = match_stats.sort_values("total_count", ascending=False).reset_index(drop=True)

# 查看结果
#print(match_stats)


In [167]:
match_stats

,cancer_type,total_count,circlebase_match_count,eccDNA_Atlas_match_count,eccDNAdb_match_count,scEccDNAdb_match_count,FISH_match_count,circlebase_match_ratio,eccDNA_Atlas_match_ratio,eccDNAdb_match_ratio,scEccDNAdb_match_ratio,FISH_match_ratio,all_match_count,all_match_ratio
0,BRCA,4098,1306,1492,636,57,334,0.318692,0.364080,0.155198,0.013909,0.081503,1530,0.373353
1,MM,362,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
2,UCEC,308,40,40,18,0,0,0.129870,0.129870,0.058442,0.000000,0.000000,42,0.136364
3,SKCM,71,27,30,15,0,3,0.380282,0.422535,0.211268,0.000000,0.042254,32,0.450704
4,PDAC,52,19,0,0,0,0,0.365385,0.000000,0.000000,0.000000,0.000000,19,0.365385
5,OV,38,17,38,13,0,20,0.447368,1.000000,0.342105,0.000000,0.526316,38,1.000000
6,CRC,29,3,6,2,0,1,0.103448,0.206897,0.068966,0.000000,0.034483,7,0.241379
7,HNSCC,8,3,5,2,0,0,0.375000,0.625000,0.250000,0.000000,0.000000,5,0.625000
8,CESC,7,2,2,2,0,0,0.285714,0.285714,0.285714,0.000000,0.000000,2,0.285714


In [195]:
######  整合后的基因情况

# Step 5: 整合“是否任一数据库有匹配”
spe_ecdna_df["all_match"] = spe_ecdna_df[match_columns].notna().any(axis=1)

# 保留至少被一个数据库匹配到的空间区域
spe_ecdna_all_match = spe_ecdna_df[spe_ecdna_df["all_match"]]
spe_ecdna_all_match

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,...,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match,all_match,FISH_match,fish_match_tf
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],False,[chr15:45778668_52964692],NaN,NaN,True,NaN,False
chr10:2100000_2200000,0.000460,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr10:2100000_2200000,BRCA,"[chr10:58487_35515702, chr10:811010_24619881]",False,"[chr10:58487_35515702, chr10:811010_24619881]",NaN,NaN,True,NaN,False
chrX:77800000_77900000,0.000235,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chrX:77800000_77900000,BRCA,NaN,False,[chrX:45727997_95519701],NaN,NaN,True,NaN,False
chr6:19600000_19700000,0.000098,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr6:19600000_19700000,BRCA,"[chr6:18310838_19863422, chr6:19305372_22564400]",False,"[chr6:18310838_19863422, chr6:19305372_22564400]",NaN,[chr6:19305603_22564629],True,NaN,False
chr18:32500000_32600000,0.000137,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr18:32500000_32600000,BRCA,[chr18:32605843_32806305],False,[chr18:32605843_32806305],NaN,NaN,True,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chr5:28100000_28200000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,...,chr5:28100000_28200000,UCEC,[chr5:15332902_29546616],False,[chr5:15332902_29546616],NaN,NaN,True,NaN,False
chr8:9500000_9600000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001042,...,chr8:9500000_9600000,UCEC,[chr8:1775777_11102156],False,[chr8:1775777_11102156],NaN,[chr8:1723943_10959666],True,NaN,False
chr12:128200000_128300000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000620,...,chr12:128200000_128300000,UCEC,[chr12:125886998_128242570],False,[chr12:125886998_128242570],NaN,NaN,True,NaN,False
chr8:92500000_92600000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,...,chr8:92500000_92600000,UCEC,[chr8:90599549_127541229],False,[chr8:90599549_127541229],NaN,NaN,True,NaN,False


In [196]:
spe_ecdna_all_match["chr_raw"] = spe_ecdna_all_match["ecdna_cpr"]

df_ecdna = spe_ecdna_all_match.copy()
df_ecdna_gene = genebody_region(df_fragments=df_ecdna,species="hg38")
df_ecdna_gene['oncogene'] = df_ecdna_gene['genebody_region_gene'].apply(custom_transform)
df_ecdna_gene

species value: hg38


,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,...,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match,all_match,FISH_match,fish_match_tf,chr_raw,genebody_region,genebody_region_gene,oncogene
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,[chr15:45778668_52964692],NaN,NaN,True,NaN,False,chr15:50100000_50200000,2,"[ATP8B4, SLC27A2]",[]
chr10:2100000_2200000,0.000460,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,"[chr10:58487_35515702, chr10:811010_24619881]",NaN,NaN,True,NaN,False,chr10:2100000_2200000,3,"[RP11-69C17.3, RP11-69C17.4, RNU6-576P]",[]
chrX:77800000_77900000,0.000235,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,[chrX:45727997_95519701],NaN,NaN,True,NaN,False,chrX:77800000_77900000,4,"[MAGT1, RNU6-854P, RN7SL460P, COX7B]",[]
chr6:19600000_19700000,0.000098,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,"[chr6:18310838_19863422, chr6:19305372_22564400]",NaN,[chr6:19305603_22564629],True,NaN,False,chr6:19600000_19700000,4,"[RP4-625H18.2, KRT18P38, UQCRFS1P3, RNU6-801P]",[]
chr18:32500000_32600000,0.000137,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,[chr18:32605843_32806305],NaN,NaN,True,NaN,False,chr18:32500000_32600000,2,"[WBP11P1, RP11-386P4.1]",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chr5:28100000_28200000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,...,[chr5:15332902_29546616],NaN,NaN,True,NaN,False,chr5:28100000_28200000,0,0,0
chr8:9500000_9600000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001042,...,[chr8:1775777_11102156],NaN,[chr8:1723943_10959666],True,NaN,False,chr8:9500000_9600000,2,"[RP11-375N15.2, TNKS]",[]
chr12:128200000_128300000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000620,...,[chr12:125886998_128242570],NaN,NaN,True,NaN,False,chr12:128200000_128300000,4,"[MIR4419B, TMEM132C, RP11-417O18.1, MIR3612]",[]
chr8:92500000_92600000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,...,[chr8:90599549_127541229],NaN,NaN,True,NaN,False,chr8:92500000_92600000,2,"[RP11-587H10.1, RP11-587H10.2]",[]


In [197]:
df_merge_body = group_merge(df_ecdna_gene,"cancer_type","genebody_region_gene")
# 计算col1列每个元素的长度，并创建新的col2列
df_merge_body['genebody_region_gene_nor_number'] = df_merge_body['genebody_region_gene_nor'].apply(lambda x: len(x))
df_merge_body

,cancer_type,genebody_region_gene_nor,most_genebody_region_gene_nor,genebody_region_gene_nor_number
0,BRCA,"[ATP8B4, SLC27A2, RP11-69C17.3, RP11-69C17.4, ...","[DLG2, ZFPM2, RBFOX1, MIR4300HG, MALRD1, RP11-...",2976
1,CESC,"[SHOX2, RSRC1, KCNMB2, RP11-385J1.3, KCNMB2-AS1]","[SHOX2, RSRC1, KCNMB2, RP11-385J1.3, KCNMB2-AS1]",5
2,CRC,"[FGF9, RN7SL766P, ZC3H13, CPB2-AS1, CPB2, CDH1...","[FGF9, RN7SL766P, ZC3H13, CPB2-AS1, CPB2, CDH1...",14
3,HNSCC,"[MECOM, RP11-641D5.2, RP11-641D5.1, AC074033.1...","[MECOM, RP11-641D5.2, RP11-641D5.1, AC074033.1...",15
4,OV,"[RP11-10O22.1, LINC01192, GPC6, GPC6-AS2, SVEP...","[RP11-10O22.1, LINC01192, GPC6, GPC6-AS2, SVEP...",79
5,PDAC,"[FAM185A, FBXL13, RNU6-1136P, RN7SKP198, DDX11...","[C9, FAM185A, FBXL13, RNU6-1136P, RN7SKP198, D...",51
6,SKCM,"[SPESP1, RP11-809H16.2, NOX5, MITF, SAMMSON, P...","[SPESP1, RP11-809H16.2, NOX5, MITF, SAMMSON, P...",52
7,UCEC,"[AC002368.4, AFF2, AFF2-IT1, BCAT1, RP11-625L1...","[CSMD1, TNKS, LINC01524, AC002368.4, AFF2, AFF...",96


In [198]:
df_merge_onco = group_merge(df_ecdna_gene,"cancer_type","oncogene")
# 计算col1列每个元素的长度，并创建新的col2列
df_merge_onco['oncogene_nor_number'] = df_merge_onco['oncogene_nor'].apply(lambda x: len(x))
df_merge_onco

,cancer_type,oncogene_nor,most_oncogene_nor,oncogene_nor_number
0,BRCA,"[SSX1, AKT3, RSPO3, RGS7, FAM135B, USP8, PTPRT...","[PTPRT, CTNND2, CSMD3, RGS7, LPP, AR, AKT3, GR...",76
1,CESC,[],[],0
2,CRC,[],[],0
3,HNSCC,[MECOM],[MECOM],1
4,OV,[],[],0
5,PDAC,[VTI1A],[VTI1A],1
6,SKCM,"[MITF, ROBO2]","[MITF, ROBO2]",2
7,UCEC,[MECOM],[MECOM],1


In [ ]:
### 反向统计
eccDNAdb_clean
eccDNA_Atlas_clean
circlebase_clean
scEccDNAdb_clean
fish_df

In [168]:
print(spe_ecdna_df.columns)

Index(['BRCA', 'CEAD', 'CESC', 'CRC', 'HNSCC', 'MM', 'OV', 'PDAC', 'SKCM',
       'UCEC', 'label', 'label_abbr', 'ecdna_cpr', 'cancer_type',
       'circlebase_match', 'has_match', 'eccDNA_Atlas_match',
       'scEccDNAdb_match', 'eccDNAdb_match', 'all_match', 'FISH_match'],
      dtype='object')


In [208]:
spe_ecdna_df

,BRCA,CEAD,CESC,CRC,HNSCC,MM,OV,PDAC,SKCM,UCEC,...,ecdna_cpr,cancer_type,circlebase_match,has_match,eccDNA_Atlas_match,scEccDNAdb_match,eccDNAdb_match,all_match,FISH_match,fish_match_tf
chr15:50100000_50200000,0.000655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr15:50100000_50200000,BRCA,[chr15:45778668_52964692],False,[chr15:45778668_52964692],NaN,NaN,True,NaN,False
chr2:176700000_176800000,0.000411,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr2:176700000_176800000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN,False
chr4:169400000_169500000,0.000225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr4:169400000_169500000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN,False
chr10:81000000_81100000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr10:81000000_81100000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN,False
chr4:98800000_98900000,0.000166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,chr4:98800000_98900000,BRCA,NaN,False,NaN,NaN,NaN,False,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:120300000_120400000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000447,...,chrX:120300000_120400000,UCEC,NaN,False,NaN,NaN,NaN,False,NaN,False
chr5:114000000_114100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000347,...,chr5:114000000_114100000,UCEC,NaN,False,NaN,NaN,NaN,False,NaN,False
chr1:214800000_214900000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000223,...,chr1:214800000_214900000,UCEC,NaN,False,NaN,NaN,NaN,False,NaN,False
chr5:35700000_35800000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000397,...,chr5:35700000_35800000,UCEC,[chr5:32261453_41811742],False,[chr5:32261453_41811742],NaN,NaN,True,NaN,False


In [136]:
def calc_reverse_match_ratio_by_name(db_df, db_name, spe_ecdna_df, match_col, region_col="ecdna_region_hg38"):
    """
    对数据库中每个癌种 sample_type_abbr 内部的 region 名称做命中统计。
    命中条件是：该 region 名称出现在 spe_ecdna_df[match_col] 中。
    """
    # 从 spe_ecdna_df 的 match_col 提取所有非空、拆分后的 region 名称（flat 列表）
    matched_regions = spe_ecdna_df[match_col].dropna().explode().tolist()
    
    # 如果 match_col 是字符串形式，需要先转列表（即按换行符或其他分隔符分割）
    if isinstance(matched_regions[0], str) and '\n' in matched_regions[0]:
        matched_regions = [
            region.strip()
            for cell in spe_ecdna_df[match_col].dropna()
            for region in cell.split('\n')
        ]
    
    matched_regions = set(matched_regions)

    # 过滤数据库，只保留在 matched_regions 中的条目
    db_df_filtered = db_df[db_df[region_col].isin(matched_regions)]

    # 筛选数据库，仅保留 match_stats 中出现过的癌种
    valid_cancer_types = spe_ecdna_df["cancer_type"].unique()
    db_df = db_df[db_df["sample_type_abbr"].isin(valid_cancer_types)]
    db_df_filtered = db_df_filtered[db_df_filtered["sample_type_abbr"].isin(valid_cancer_types)]

    # 每个癌种的总数和命中数
    total = db_df.groupby("sample_type_abbr")[region_col].count().rename("total")
    matched = db_df_filtered.groupby("sample_type_abbr")[region_col].count().rename("matched")

    # 合并计算命中率
    df = pd.concat([total, matched], axis=1).fillna(0)
    df["reverse_match_ratio"] = df["matched"] / df["total"]
    df.columns = [f"{db_name}_total", f"{db_name}_matched", f"{db_name}_reverse_match_ratio"]

    return df


In [173]:
circlebase_reverse = calc_reverse_match_ratio_by_name(
    db_df=circlebase_clean,
    db_name="circlebase",
    spe_ecdna_df=spe_ecdna_df,
    match_col="circlebase_match"
)

eccDNA_Atlas_reverse = calc_reverse_match_ratio_by_name(
    db_df=eccDNA_Atlas_clean,
    db_name="eccDNA_Atlas",
    spe_ecdna_df=spe_ecdna_df,
    match_col="eccDNA_Atlas_match"
)

scEccDNAdb_reverse = calc_reverse_match_ratio_by_name(
    db_df=scEccDNAdb_clean,
    db_name="scEccDNAdb",
    spe_ecdna_df=spe_ecdna_df,
    match_col="scEccDNAdb_match"
)

eccDNAdb_reverse = calc_reverse_match_ratio_by_name(
    db_df=eccDNAdb_clean,
    db_name="eccDNAdb",
    spe_ecdna_df=spe_ecdna_df,
    match_col="eccDNAdb_match"
)

In [174]:
FISH_reverse = calc_reverse_match_ratio_by_name(
    db_df=fish_df,
    db_name="FISH",
    spe_ecdna_df=spe_ecdna_df,
    match_col="FISH_match"
)

In [175]:
reverse_stats = pd.concat([
    circlebase_reverse,
    eccDNA_Atlas_reverse,
    scEccDNAdb_reverse,
    eccDNAdb_reverse,
    FISH_reverse
], axis=1)

reverse_stats.reset_index(inplace=True)
reverse_stats.rename(columns={"sample_type_abbr": "cancer_type"}, inplace=True)

In [176]:
reverse_stats

,cancer_type,circlebase_total,circlebase_matched,circlebase_reverse_match_ratio,eccDNA_Atlas_total,eccDNA_Atlas_matched,eccDNA_Atlas_reverse_match_ratio,scEccDNAdb_total,scEccDNAdb_matched,scEccDNAdb_reverse_match_ratio,eccDNAdb_total,eccDNAdb_matched,eccDNAdb_reverse_match_ratio,FISH_total,FISH_matched,FISH_reverse_match_ratio
0,BRCA,723.0,381.0,0.526971,1103,560.0,0.507706,32.0,14.0,0.4375,447.0,175.0,0.391499,79.0,47.0,0.594937
1,CESC,49.0,3.0,0.061224,49,3.0,0.061224,3.0,0.0,0.0000,16.0,2.0,0.125000,NaN,NaN,NaN
2,CRC,77.0,2.0,0.025974,204,6.0,0.029412,180.0,0.0,0.0000,367.0,2.0,0.005450,73.0,1.0,0.013699
3,HNSCC,294.0,3.0,0.010204,10405,9.0,0.000865,NaN,NaN,NaN,77.0,2.0,0.025974,NaN,NaN,NaN
4,OV,316.0,18.0,0.056962,173335,365.0,0.002106,4.0,0.0,0.0000,110.0,7.0,0.063636,576.0,20.0,0.034722
5,PDAC,144.0,16.0,0.111111,13,0.0,0.000000,NaN,NaN,NaN,2.0,0.0,0.000000,NaN,NaN,NaN
6,SKCM,418.0,32.0,0.076555,476,35.0,0.073529,NaN,NaN,NaN,180.0,15.0,0.083333,15.0,2.0,0.133333
7,UCEC,242.0,24.0,0.099174,270,25.0,0.092593,NaN,NaN,NaN,69.0,10.0,0.144928,NaN,NaN,NaN
8,MM,NaN,NaN,NaN,2,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [177]:
FISH_reverse_match = reverse_stats[["cancer_type","FISH_total","FISH_matched","FISH_reverse_match_ratio"]]
FISH_reverse_match

,cancer_type,FISH_total,FISH_matched,FISH_reverse_match_ratio
0,BRCA,79.0,47.0,0.594937
1,CESC,NaN,NaN,NaN
2,CRC,73.0,1.0,0.013699
3,HNSCC,NaN,NaN,NaN
4,OV,576.0,20.0,0.034722
5,PDAC,NaN,NaN,NaN
6,SKCM,15.0,2.0,0.133333
7,UCEC,NaN,NaN,NaN
8,MM,NaN,NaN,NaN


In [ ]:
##### 基于各个 cancer 的全部 ecDNA 计算 FISH overlap 

In [202]:
df_cancer_sample_cpr = pd.read_csv('./fig_100k_3/cell_frequency/f00-df_cancer_sample_cpr.xls', sep='\t')
df_cancer_sample_cpr

,cancer,sample,ecDNA_CPR,cell_number_cov6pos,cell_frequency_cov6pos,Ave_coverage_in_sample,Max_coverage_in_sample,Median_coverage_in_sample,chr_100k,n_cells
0,BRCA,ht029b1-s1pc,chr10:10000000_10100000,3,0.000817,0.006475,8.891147,0.0,chr10:10000000_10100000,3.0
1,BRCA,ht029b1-s1pc,chr10:1000000_1100000,215,0.058583,0.500815,39.768020,0.0,chr10:1000000_1100000,215.0
2,BRCA,ht029b1-s1pc,chr10:100000_200000,18,0.004905,0.040703,17.863032,0.0,chr10:100000_200000,18.0
3,BRCA,ht029b1-s1pc,chr10:100100000_100200000,3,0.000817,0.005242,6.622517,0.0,chr10:100100000_100200000,3.0
4,BRCA,ht029b1-s1pc,chr10:100300000_100400000,5,0.001362,0.011711,13.365410,0.0,chr10:100300000_100400000,5.0
...,...,...,...,...,...,...,...,...,...,...
765038,UCEC,cpt704du-t1,chrX:153900000_154000000,6,0.011650,0.101160,13.002601,0.0,chrX:153900000_154000000,6.0
765039,UCEC,cpt704du-t1,chrX:154300000_154400000,3,0.005825,0.050279,10.220470,0.0,chrX:154300000_154400000,3.0
765040,UCEC,cpt704du-t1,chrX:154400000_154500000,5,0.009709,0.099661,14.542810,0.0,chrX:154400000_154500000,5.0
765041,UCEC,cpt704du-t1,chrX:49100000_49200000,4,0.007767,0.060018,9.803922,0.0,chrX:49100000_49200000,4.0


In [203]:
df_cancer_sample_cpr["cell_number_cov6pos"].min()

3

In [205]:
dftmp = df_cancer_sample_cpr[["cancer","sample","ecDNA_CPR"]]

# 使用 groupby 聚合，并统计每个 ecDNA_CPR 在每个 cancer 中出现的样本数（去重 sample）
result = (
    dftmp.groupby(['ecDNA_CPR', 'cancer'])['sample']
    .nunique()  # 统计唯一 sample 数量
    .reset_index(name='sample_count')  # 重命名结果列
)

# 显示前几行结果
print(result.head())

                   ecDNA_CPR cancer  sample_count
0             chr10:0_100000   BRCA             3
1             chr10:0_100000     MM             1
2             chr10:0_100000     OV             1
3             chr10:0_100000   UCEC             1
4  chr10:100000000_100100000   BRCA             7


In [211]:
result["ecdna_cpr"] = result["ecDNA_CPR"]
result["cancer_type"] = result["cancer"]
result

,ecDNA_CPR,cancer,sample_count,ecdna_cpr,cancer_type
0,chr10:0_100000,BRCA,3,chr10:0_100000,BRCA
1,chr10:0_100000,MM,1,chr10:0_100000,MM
2,chr10:0_100000,OV,1,chr10:0_100000,OV
3,chr10:0_100000,UCEC,1,chr10:0_100000,UCEC
4,chr10:100000000_100100000,BRCA,7,chr10:100000000_100100000,BRCA
...,...,...,...,...,...
161708,chrY:8800000_8900000,MM,1,chrY:8800000_8900000,MM
161709,chrY:8900000_9000000,MM,1,chrY:8900000_9000000,MM
161710,chrY:9200000_9300000,MM,1,chrY:9200000_9300000,MM
161711,chrY:9500000_9600000,MM,1,chrY:9500000_9600000,MM


In [212]:
result2 = result[result["sample_count"]>=2]
result2

,ecDNA_CPR,cancer,sample_count,ecdna_cpr,cancer_type
0,chr10:0_100000,BRCA,3,chr10:0_100000,BRCA
4,chr10:100000000_100100000,BRCA,7,chr10:100000000_100100000,BRCA
5,chr10:100000000_100100000,CESC,4,chr10:100000000_100100000,CESC
6,chr10:100000000_100100000,CRC,5,chr10:100000000_100100000,CRC
9,chr10:100000000_100100000,OV,3,chr10:100000000_100100000,OV
...,...,...,...,...,...
161682,chrY:56800000_56900000,SKCM,2,chrY:56800000_56900000,SKCM
161683,chrY:56800000_56900000,UCEC,4,chrY:56800000_56900000,UCEC
161686,chrY:7000000_7100000,MM,2,chrY:7000000_7100000,MM
161704,chrY:8600000_8700000,MM,2,chrY:8600000_8700000,MM


In [213]:
result3 = result[result["sample_count"]>=3]
result3

,ecDNA_CPR,cancer,sample_count,ecdna_cpr,cancer_type
0,chr10:0_100000,BRCA,3,chr10:0_100000,BRCA
4,chr10:100000000_100100000,BRCA,7,chr10:100000000_100100000,BRCA
5,chr10:100000000_100100000,CESC,4,chr10:100000000_100100000,CESC
6,chr10:100000000_100100000,CRC,5,chr10:100000000_100100000,CRC
9,chr10:100000000_100100000,OV,3,chr10:100000000_100100000,OV
...,...,...,...,...,...
161569,chrX:9900000_10000000,UCEC,3,chrX:9900000_10000000,UCEC
161581,chrX:99600000_99700000,BRCA,3,chrX:99600000_99700000,BRCA
161679,chrY:56800000_56900000,MM,4,chrY:56800000_56900000,MM
161680,chrY:56800000_56900000,OV,4,chrY:56800000_56900000,OV


In [ ]:
#######   all_ecdna_df  代替 ：： spe_ecdna_df

In [217]:
import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean

ref_db = fish_df.copy()

# 复制索引为新列
all_ecdna_df = result3.copy()

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in all_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
all_ecdna_df["FISH_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
#print(all_ecdna_df[["ecdna_cpr", "cancer_type", "FISH_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
all_ecdna_df["has_match"] = all_ecdna_df["FISH_match"].notna()

# 按 cancer_type 分组，统计匹配比例
all_match_stats = all_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
all_match_stats["match_ratio"] = all_match_stats["match_count"] / all_match_stats["total_count"]

# 查看统计结果
print(all_match_stats)

all_ecdna_df

             total_count  match_count  match_ratio
cancer_type                                       
BRCA               17665         1862     0.105406
CEAD                3037            0     0.000000
CESC                7695            0     0.000000
CRC                 8284          400     0.048286
HNSCC               6729            0     0.000000
MM                  4870            0     0.000000
OV                  8504         4175     0.490945
PDAC                9730            0     0.000000
SKCM                8287          182     0.021962
UCEC               11937            0     0.000000


,ecDNA_CPR,cancer,sample_count,ecdna_cpr,cancer_type,FISH_match,has_match
0,chr10:0_100000,BRCA,3,chr10:0_100000,BRCA,NaN,False
4,chr10:100000000_100100000,BRCA,7,chr10:100000000_100100000,BRCA,NaN,False
5,chr10:100000000_100100000,CESC,4,chr10:100000000_100100000,CESC,NaN,False
6,chr10:100000000_100100000,CRC,5,chr10:100000000_100100000,CRC,NaN,False
9,chr10:100000000_100100000,OV,3,chr10:100000000_100100000,OV,"[chr10:72278212_113357713, chr10:94220365_1074...",True
...,...,...,...,...,...,...,...
161569,chrX:9900000_10000000,UCEC,3,chrX:9900000_10000000,UCEC,NaN,False
161581,chrX:99600000_99700000,BRCA,3,chrX:99600000_99700000,BRCA,NaN,False
161679,chrY:56800000_56900000,MM,4,chrY:56800000_56900000,MM,NaN,False
161680,chrY:56800000_56900000,OV,4,chrY:56800000_56900000,OV,NaN,False


In [218]:
import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean
fish_df
eccDNA_Atlas_clean
circlebase_clean
eccDNAdb_clean
scEccDNAdb_clean

ref_db = eccDNA_Atlas_clean.copy()

# 复制索引为新列
#all_ecdna_df = result3.copy()

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in all_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
all_ecdna_df["eccDNA_Atlas_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
#print(all_ecdna_df[["ecdna_cpr", "cancer_type", "eccDNA_Atlas_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
all_ecdna_df["has_match"] = all_ecdna_df["eccDNA_Atlas_match"].notna()

# 按 cancer_type 分组，统计匹配比例
all_match_stats = all_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
all_match_stats["match_ratio"] = all_match_stats["match_count"] / all_match_stats["total_count"]

# 查看统计结果
print(all_match_stats)

             total_count  match_count  match_ratio
cancer_type                                       
BRCA               17665         6567     0.371752
CEAD                3037         2944     0.969378
CESC                7695          606     0.078752
CRC                 8284          974     0.117576
HNSCC               6729         4355     0.647199
MM                  4870            2     0.000411
OV                  8504         8502     0.999765
PDAC                9730           29     0.002980
SKCM                8287         2711     0.327139
UCEC               11937         1881     0.157577


In [219]:
import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean
fish_df
eccDNA_Atlas_clean
circlebase_clean
eccDNAdb_clean
scEccDNAdb_clean

ref_db = circlebase_clean.copy()

# 复制索引为新列
#all_ecdna_df = result3.copy()

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in all_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
all_ecdna_df["circlebase_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
#print(all_ecdna_df[["ecdna_cpr", "cancer_type", "circlebase_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
all_ecdna_df["has_match"] = all_ecdna_df["circlebase_match"].notna()

# 按 cancer_type 分组，统计匹配比例
all_match_stats = all_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
all_match_stats["match_ratio"] = all_match_stats["match_count"] / all_match_stats["total_count"]

# 查看统计结果
print(all_match_stats)

             total_count  match_count  match_ratio
cancer_type                                       
BRCA               17665         5880     0.332862
CEAD                3037            0     0.000000
CESC                7695          606     0.078752
CRC                 8284          342     0.041284
HNSCC               6729          890     0.132263
MM                  4870            0     0.000000
OV                  8504         3551     0.417568
PDAC                9730         3941     0.405036
SKCM                8287         2473     0.298419
UCEC               11937         1701     0.142498


In [220]:
import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean
fish_df
eccDNA_Atlas_clean
circlebase_clean
eccDNAdb_clean
scEccDNAdb_clean

ref_db = eccDNAdb_clean.copy()

# 复制索引为新列
#all_ecdna_df = result3.copy()

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in all_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
all_ecdna_df["eccDNAdb_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
#print(all_ecdna_df[["ecdna_cpr", "cancer_type", "eccDNAdb_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
all_ecdna_df["has_match"] = all_ecdna_df["eccDNAdb_match"].notna()

# 按 cancer_type 分组，统计匹配比例
all_match_stats = all_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
all_match_stats["match_ratio"] = all_match_stats["match_count"] / all_match_stats["total_count"]

# 查看统计结果
print(all_match_stats)

             total_count  match_count  match_ratio
cancer_type                                       
BRCA               17665         2809     0.159015
CEAD                3037            0     0.000000
CESC                7695          257     0.033398
CRC                 8284          333     0.040198
HNSCC               6729          560     0.083222
MM                  4870            0     0.000000
OV                  8504         1901     0.223542
PDAC                9730            4     0.000411
SKCM                8287         1615     0.194884
UCEC               11937          665     0.055709


In [221]:
import pandas as pd
import numpy as np

# --------------------- Step 0：准备输入数据 ---------------------
# 示例：你的输入 DataFrame 是 spe_ecdna_df 和 circlebase_clean
fish_df
eccDNA_Atlas_clean
circlebase_clean
eccDNAdb_clean
scEccDNAdb_clean

ref_db = scEccDNAdb_clean.copy()

# 复制索引为新列
#all_ecdna_df = result3.copy()

# --------------------- Step 1：优化后的 overlap 函数 ---------------------
def parse_region(region_str):
    chrom, coords = region_str.split(':')
    start_str, end_str = coords.split('_')
    return chrom, int(start_str), int(end_str)
def parse_region(region_str):
    try:
        parts = region_str.split(':')
        if len(parts) != 2:
            raise ValueError(f"Region format error: '{region_str}'")
        chrom, coords = parts
        start_str, end_str = coords.split('_')
        return chrom, int(start_str), int(end_str)
    except Exception as e:
        print(f"[parse_region] Invalid region format: '{region_str}' → {e}")
        raise

def has_overlap(region1, region2, max_distance=50000):
    chr1, start1, end1 = parse_region(region1)
    chr2, start2, end2 = parse_region(region2)
    if chr1 != chr2:
        return False
    return not (end1 + max_distance < start2 or start1 - max_distance > end2)

# --------------------- Step 2：确保 circlebase_clean 有正确区域列 ---------------------
if "ecdna_region_hg38" not in ref_db.columns:
    print("ecdna_region_hg38:NULL")

# --------------------- Step 3：逐行比对，并生成匹配结果 ---------------------
# 初始化匹配结果列
match_results = []

for idx, row in all_ecdna_df.iterrows():
    ecdna_cpr0 = row["ecdna_cpr"]
    cancer_type0 = row["cancer_type"]
    
    # 获取数据库中与 cancer_type0 匹配的 ecDNA 区域集合 S1
    matched_rows = ref_db[ref_db["sample_type_abbr"] == cancer_type0]
    S1 = matched_rows["ecdna_region_hg38"].tolist()
    #print(S1)
    
    # 比较交集，找出与 ecdna_cpr0 有重叠的 S1 中的 region
    S2 = [region for region in S1 if has_overlap(ecdna_cpr0, region, max_distance=50000)]
    
    # 记录匹配结果（如果无匹配，则为 np.nan）
    match_results.append(S2 if S2 else np.nan)

# 添加新列到 DataFrame
all_ecdna_df["scEccDNAdb_match"] = match_results

# --------------------- Step 4：结果查看 ---------------------
#print(all_ecdna_df[["ecdna_cpr", "cancer_type", "scEccDNAdb_match"]].head())

# --------------------- Step 5：统计 ---------------------
# 将匹配情况转为布尔值（是否非空匹配）
all_ecdna_df["has_match"] = all_ecdna_df["scEccDNAdb_match"].notna()

# 按 cancer_type 分组，统计匹配比例
all_match_stats = all_ecdna_df.groupby("cancer_type")["has_match"].agg([
    ("total_count", "count"),
    ("match_count", "sum")
])

# 计算占比（百分比）
all_match_stats["match_ratio"] = all_match_stats["match_count"] / all_match_stats["total_count"]

# 查看统计结果
print(all_match_stats)

             total_count  match_count  match_ratio
cancer_type                                       
BRCA               17665          197     0.011152
CEAD                3037            0     0.000000
CESC                7695            1     0.000130
CRC                 8284           45     0.005432
HNSCC               6729            0     0.000000
MM                  4870            0     0.000000
OV                  8504            1     0.000118
PDAC                9730            0     0.000000
SKCM                8287            0     0.000000
UCEC               11937            0     0.000000


In [222]:
all_ecdna_df

,ecDNA_CPR,cancer,sample_count,ecdna_cpr,cancer_type,FISH_match,has_match,eccDNA_Atlas_match,circlebase_match,eccDNAdb_match,scEccDNAdb_match
0,chr10:0_100000,BRCA,3,chr10:0_100000,BRCA,NaN,False,[chr10:58487_35515702],[chr10:58487_35515702],NaN,NaN
4,chr10:100000000_100100000,BRCA,7,chr10:100000000_100100000,BRCA,NaN,False,NaN,NaN,NaN,NaN
5,chr10:100000000_100100000,CESC,4,chr10:100000000_100100000,CESC,NaN,False,NaN,NaN,NaN,NaN
6,chr10:100000000_100100000,CRC,5,chr10:100000000_100100000,CRC,NaN,False,NaN,NaN,NaN,NaN
9,chr10:100000000_100100000,OV,3,chr10:100000000_100100000,OV,"[chr10:72278212_113357713, chr10:94220365_1074...",False,"[chr10:50178812_176448517, chr10:99960515_9996...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
161569,chrX:9900000_10000000,UCEC,3,chrX:9900000_10000000,UCEC,NaN,False,NaN,NaN,NaN,NaN
161581,chrX:99600000_99700000,BRCA,3,chrX:99600000_99700000,BRCA,NaN,False,NaN,NaN,NaN,NaN
161679,chrY:56800000_56900000,MM,4,chrY:56800000_56900000,MM,NaN,False,NaN,NaN,NaN,NaN
161680,chrY:56800000_56900000,OV,4,chrY:56800000_56900000,OV,NaN,False,"[chrY:56861301_56861514, chrY:56865347_5686546...",NaN,NaN,NaN


In [226]:
# Step 1: 基础统计（每种 cancer_type 的总样本数）
all_match_stats = all_ecdna_df.groupby("cancer_type").size().reset_index(name="total_count")

# Step 2: 定义四个数据库的匹配列
match_columns = [
    "FISH_match",
    "circlebase_match",
    "eccDNA_Atlas_match",
    "eccDNAdb_match",
    "scEccDNAdb_match"
]

# Step 3: 分别统计四个数据库的匹配 count 和 ratio
for col in match_columns:
    tmp = all_ecdna_df.copy()
    tmp["has_match"] = tmp[col].notna()
    
    stats = (
        tmp.groupby("cancer_type")["has_match"]
        .agg(match_count="sum")
        .reset_index()
        .rename(columns={"match_count": f"{col}_count"})
    )
    
    all_match_stats = all_match_stats.merge(stats, on="cancer_type", how="left")

# Step 4: 计算匹配比例
for col in match_columns:
    all_match_stats[f"{col}_ratio"] = all_match_stats[f"{col}_count"] / all_match_stats["total_count"]

# Step 5: 整合“是否任一数据库有匹配”
all_ecdna_df["all_match"] = all_ecdna_df[match_columns].notna().any(axis=1)

# Step 6: 统计整合后的匹配情况
all_match_all_stats = (
    all_ecdna_df.groupby("cancer_type")["all_match"]
    .agg(all_match_count="sum")
    .reset_index()
)
all_match_stats = all_match_stats.merge(all_match_all_stats, on="cancer_type", how="left")
all_match_stats["all_match_ratio"] = all_match_stats["all_match_count"] / all_match_stats["total_count"]

# Step 7: 排序 & 展示
all_match_stats = all_match_stats.sort_values("total_count", ascending=False).reset_index(drop=True)

# 查看结果
all_match_stats

,cancer_type,total_count,FISH_match_count,circlebase_match_count,eccDNA_Atlas_match_count,eccDNAdb_match_count,scEccDNAdb_match_count,FISH_match_ratio,circlebase_match_ratio,eccDNA_Atlas_match_ratio,eccDNAdb_match_ratio,scEccDNAdb_match_ratio,all_match_count,all_match_ratio
0,BRCA,17665,1862,5880,6567,2809,197,0.105406,0.332862,0.371752,0.159015,0.011152,6726,0.380753
1,UCEC,11937,0,1701,1881,665,0,0.000000,0.142498,0.157577,0.055709,0.000000,1963,0.164447
2,PDAC,9730,0,3941,29,4,0,0.000000,0.405036,0.002980,0.000411,0.000000,3970,0.408016
3,OV,8504,4175,3551,8502,1901,1,0.490945,0.417568,0.999765,0.223542,0.000118,8502,0.999765
4,SKCM,8287,182,2473,2711,1615,0,0.021962,0.298419,0.327139,0.194884,0.000000,3007,0.362857
5,CRC,8284,400,342,974,333,45,0.048286,0.041284,0.117576,0.040198,0.005432,1037,0.125181
6,CESC,7695,0,606,606,257,1,0.000000,0.078752,0.078752,0.033398,0.000130,616,0.080052
7,HNSCC,6729,0,890,4355,560,0,0.000000,0.132263,0.647199,0.083222,0.000000,4461,0.662951
8,MM,4870,0,0,2,0,0,0.000000,0.000000,0.000411,0.000000,0.000000,2,0.000411
9,CEAD,3037,0,0,2944,0,0,0.000000,0.000000,0.969378,0.000000,0.000000,2944,0.969378


In [227]:
all_match_stats_fish = all_match_stats[["cancer_type","total_count","FISH_match_count","FISH_match_ratio","all_match_count","all_match_ratio"]]
all_match_stats_fish

,cancer_type,total_count,FISH_match_count,FISH_match_ratio,all_match_count,all_match_ratio
0,BRCA,17665,1862,0.105406,6726,0.380753
1,UCEC,11937,0,0.000000,1963,0.164447
2,PDAC,9730,0,0.000000,3970,0.408016
3,OV,8504,4175,0.490945,8502,0.999765
4,SKCM,8287,182,0.021962,3007,0.362857
5,CRC,8284,400,0.048286,1037,0.125181
6,CESC,7695,0,0.000000,616,0.080052
7,HNSCC,6729,0,0.000000,4461,0.662951
8,MM,4870,0,0.000000,2,0.000411
9,CEAD,3037,0,0.000000,2944,0.969378


In [215]:
all_FISH_reverse = calc_reverse_match_ratio_by_name(
    db_df=fish_df,
    db_name="FISH",
    spe_ecdna_df=all_ecdna_df,
    match_col="FISH_match"
)
all_FISH_reverse

,FISH_total,FISH_matched,FISH_reverse_match_ratio
sample_type_abbr,,,
BRCA,79,74,0.936709
CRC,73,62,0.849315
OV,576,436,0.756944
SKCM,15,13,0.866667


In [224]:
all_circlebase_reverse = calc_reverse_match_ratio_by_name(
    db_df=circlebase_clean,
    db_name="circlebase",
    spe_ecdna_df=all_ecdna_df,
    match_col="circlebase_match"
)

all_eccDNA_Atlas_reverse = calc_reverse_match_ratio_by_name(
    db_df=eccDNA_Atlas_clean,
    db_name="eccDNA_Atlas",
    spe_ecdna_df=all_ecdna_df,
    match_col="eccDNA_Atlas_match"
)

all_scEccDNAdb_reverse = calc_reverse_match_ratio_by_name(
    db_df=scEccDNAdb_clean,
    db_name="scEccDNAdb",
    spe_ecdna_df=all_ecdna_df,
    match_col="scEccDNAdb_match"
)

all_eccDNAdb_reverse = calc_reverse_match_ratio_by_name(
    db_df=eccDNAdb_clean,
    db_name="eccDNAdb",
    spe_ecdna_df=all_ecdna_df,
    match_col="eccDNAdb_match"
)

In [225]:
all_reverse_stats = pd.concat([
    all_FISH_reverse,
    all_eccDNA_Atlas_reverse,
    all_circlebase_reverse,
    all_eccDNAdb_reverse,
    all_scEccDNAdb_reverse
], axis=1)

all_reverse_stats.reset_index(inplace=True)
all_reverse_stats.rename(columns={"sample_type_abbr": "cancer_type"}, inplace=True)
all_reverse_stats

,cancer_type,FISH_total,FISH_matched,FISH_reverse_match_ratio,eccDNA_Atlas_total,eccDNA_Atlas_matched,eccDNA_Atlas_reverse_match_ratio,circlebase_total,circlebase_matched,circlebase_reverse_match_ratio,eccDNAdb_total,eccDNAdb_matched,eccDNAdb_reverse_match_ratio,scEccDNAdb_total,scEccDNAdb_matched,scEccDNAdb_reverse_match_ratio
0,BRCA,79.0,74.0,0.936709,1103,1001,0.907525,723.0,647.0,0.894882,447.0,358.0,0.800895,32.0,29.0,0.906250
1,CRC,73.0,62.0,0.849315,204,141,0.691176,77.0,38.0,0.493506,367.0,282.0,0.768392,180.0,68.0,0.377778
2,OV,576.0,436.0,0.756944,173335,115010,0.663513,316.0,234.0,0.740506,110.0,68.0,0.618182,4.0,1.0,0.250000
3,SKCM,15.0,13.0,0.866667,476,297,0.623950,418.0,257.0,0.614833,180.0,117.0,0.650000,NaN,NaN,NaN
4,CEAD,NaN,NaN,NaN,38876,16012,0.411874,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CESC,NaN,NaN,NaN,49,42,0.857143,49.0,42.0,0.857143,16.0,10.0,0.625000,3.0,3.0,1.000000
6,HNSCC,NaN,NaN,NaN,10405,5199,0.499664,294.0,166.0,0.564626,77.0,42.0,0.545455,NaN,NaN,NaN
7,MM,NaN,NaN,NaN,2,1,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,PDAC,NaN,NaN,NaN,13,5,0.384615,144.0,120.0,0.833333,2.0,1.0,0.500000,NaN,NaN,NaN
9,UCEC,NaN,NaN,NaN,270,239,0.885185,242.0,215.0,0.888430,69.0,62.0,0.898551,NaN,NaN,NaN


In [ ]:
1. 人的 ecDNA 的已验证/bulk 方法检出的ecDNA部分，inferECC复现的结果。整体验证。作为延伸。
2. 